# Wildlife Re-Identification — three models

| Model | Approach | Output CSV | Kaggle public |
|---|---|---|---|
| 1 | CSLS + Jaccard retrieval, SIFT, query graph, two-stage LightGBM, 5-member ensemble | `model1_improved_ensemble.csv` | 0.86002 |
| 2 | XQDA + KDE / two-hop retrieval, validation-consistent features, 5-member ensemble | `model2_xqda_kde_ensemble.csv` | |
| 3 | V8 fusion: PCA + LDA retrieval, SIFT, two-stage LightGBM, 9-member ensemble | `model3_v8_fusion_ensemble.csv` | 0.85677 |

Only the provided features are used (MiewID vectors, SIFT keypoints/descriptors, the CSVs) with the allowed libraries; no images, pre-trained models or external data.

**How to run:** run the package cell below, then the models in order. Each model loads the data itself and keeps its own caches (Model 1: `Work/sift_rich_cache.pkl`, Model 2: `Work/m2_*.pkl`, Model 3: `Work/Model3_V8/`). A reset cell after Models 1 and 2 frees memory, so each model starts clean. To run a single model, run the package cell and then that model's cells. All three together take roughly 1.5–2.5 hours on a CPU (Model 1 took 25 minutes in its last run).

In [ ]:
import sys
import subprocess
import importlib.util

# Allowed competition libraries: (import_name, pip_package_name)
REQUIRED_PACKAGES = [
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("scipy", "scipy"),
    ("sklearn", "scikit-learn"),
    ("lightgbm", "lightgbm"),
    ("xgboost", "xgboost"),
    ("catboost", "catboost"),
    ("cv2", "opencv-python"),
    ("networkx", "networkx"),
]

missing = [pip_name for mod_name, pip_name in REQUIRED_PACKAGES if importlib.util.find_spec(mod_name) is None]
if missing:
    print(f"Installing missing packages: {missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("Installation complete.")
else:
    print("All required packages are already installed.")

## Model 1 — Anti-Overfitting High-MAP@5 Pipeline (Kaggle public 0.86002)

**Strictly compliant with all competition rules** (only `numpy`, `pandas`, `scipy`, `scikit-learn`, `lightgbm`, `xgboost`, `catboost`, `opencv-python`, `networkx`; only provided MiewID & SIFT features; no deep learning, no external data, no raw images).

### Key Upgrades Over the Previous Pipeline (0.85677 LB)
1. **Power-Law Regularized Whitening ($\alpha = 0.5$) + Multi-Space Embeddings**: Replaces noise-amplifying full PCA whitening (`whiten=True`) with eigenvalue power-whitening, combined with Regularized LDA and L2-normalized identity centroids.
2. **Hubness Reduction (CSLS) & k-Reciprocal Jaccard Reranking**: Pure distance-based retrieval features that eliminate high-dimensional hubness and dramatically improve both top-5 ranking and `new_individual` separation.
3. **Upgraded Geometric SIFT Verification (5-Feature Affine + Sanity Check)**: Faster pre-computed `float16` RootSIFT, sanity-checked `estimateAffine2D` (handles curved flanks while rejecting degenerate warps), plus **inlier ratio**, **mean inlier similarity**, and **spatial spread** (kills clustered background false matches).
4. **Full-Graph Stage-2 Transductive Propagation (`networkx`)**: Removes the 50% test-set split from the old `Stage2.py` (which threw away half of all test-to-test same-animal links) and uses thresholded graph components that are density-consistent between CV and Test.
5. **Anti-Overfitting Convex Log-Loss Blending + Per-Species Prior-Shift EM**: Replaces 18-parameter coordinate ascent on discrete MAP@5 with smooth simplex log-loss calibration and per-species Bayes prior-shift adjustment.

**Target Runtime**: ~4–8 minutes end-to-end on CPU (SIFT features auto-cache to `Work/sift_rich_cache.pkl` so re-runs take <2 minutes).

In [ ]:
import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_v, "1")

import gc
import time
import pickle
import warnings
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import cv2
import numpy as np
import pandas as pd
import networkx as nx
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
from scipy.optimize import minimize
from scipy.special import expit, logit

from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier, CatBoostRanker

warnings.filterwarnings("ignore")
cv2.setNumThreads(1)

T_START = time.time()
NEW = "new_individual"
SEED = 42
N_WORKERS = max(1, min(8, os.cpu_count() or 4))

# ---- Auto-detect data directory (works on Kaggle, Colab, and local Windows/Linux) ----
BASE_DIR = os.environ.get("REID_BASE", None)  # Optional manual override

def resolve_paths(base_override=None):
    candidates = []
    if base_override:
        candidates.append(Path(base_override))
    candidates += [
        Path.cwd(),
        Path.cwd() / "reid_student_data_main",
        Path.cwd() / "projectwild3",
        Path.cwd() / "projectwild3" / "reid_student_data_main",
        Path("/kaggle/input"),
    ]
    # Also search immediate subdirs of /kaggle/input if present
    ki = Path("/kaggle/input")
    if ki.exists():
        for p in ki.rglob("train.csv"):
            candidates.insert(0, p.parent)

    data_dir = None
    for c in candidates:
        if (c / "train.csv").exists() and (c / "miewid_features.npy").exists():
            data_dir = c
            break
        if (c / "reid_student_data_main" / "train.csv").exists():
            data_dir = c / "reid_student_data_main"
            break
    if data_dir is None:
        raise FileNotFoundError(
            "Could not locate train.csv and miewid_features.npy. "
            "Set BASE_DIR = '/path/to/your/folder' above."
        )

    def find_sift(name):
        for p in [data_dir / name, data_dir.parent / name, Path.cwd() / name]:
            if p.exists():
                return p
        # Search upward/siblings on Kaggle
        for p in data_dir.parent.rglob(name):
            return p
        raise FileNotFoundError(f"Could not locate {name}")

    work_dir = Path("/kaggle/working/Work") if Path("/kaggle/working").exists() else (Path.cwd() / "Work")
    work_dir.mkdir(parents=True, exist_ok=True)
    out_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
    return data_dir, find_sift("sift_features_train.npz"), find_sift("sift_features_test.npz"), work_dir, out_dir

DATA_DIR, SIFT_TRAIN_PATH, SIFT_TEST_PATH, WORK_DIR, OUT_DIR = resolve_paths(BASE_DIR)
print(f"DATA_DIR:   {DATA_DIR}")
print(f"SIFT_TRAIN: {SIFT_TRAIN_PATH}")
print(f"SIFT_TEST:  {SIFT_TEST_PATH}")
print(f"WORK_DIR:   {WORK_DIR} | WORKERS: {N_WORKERS}")

In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
for df in (train, test):
    df["viewpoint"] = df["viewpoint"].fillna("unknown").astype(str)

all_meta = pd.concat([train[["id", "species", "viewpoint"]],
                      test[["id", "species", "viewpoint"]]], ignore_index=True)
SP_MAP = dict(zip(all_meta["id"], all_meta["species"]))
VP_MAP = dict(zip(all_meta["id"], all_meta["viewpoint"]))
LAB_MAP = dict(zip(train["id"], train["identity"]))
SPECIES_LIST = sorted(train["species"].unique().tolist())

F_RAW = np.load(DATA_DIR / "miewid_features.npy").astype(np.float32)
MIEW_IDS = pd.read_csv(DATA_DIR / "miewid_ids.csv")["id"].tolist()
assert len(MIEW_IDS) == len(F_RAW), "miewid_ids.csv and miewid_features.npy mismatch!"
ROW_MAP = {img_id: idx for idx, img_id in enumerate(MIEW_IDS)}
F_UNIT = F_RAW / (np.linalg.norm(F_RAW, axis=1, keepdims=True) + 1e-12)

SIDES = {
    "left": "L", "frontleft": "L", "backleft": "L",
    "right": "R", "frontright": "R", "backright": "R",
}
SIDE_CODE = {"L": 0, "R": 1, "O": 2, "U": 3}

def get_side(vp):
    return "U" if vp == "unknown" else SIDES.get(vp, "O")

def view_relation(vp_a, vp_b):
    # 0: same flank, 1: flank vs other, 2: opposite flanks (L vs R), 3: unknown
    sa, sb = get_side(vp_a), get_side(vp_b)
    if "U" in (sa, sb):
        return 3
    if sa == sb:
        return 0
    return 2 if {sa, sb} == {"L", "R"} else 1

def map5_score(truth_dict, preds_dict):
    tot = 0.0
    for q, t in truth_dict.items():
        for k, lab in enumerate(preds_dict.get(q, [])[:5]):
            if lab == t:
                tot += 1.0 / (k + 1)
                break
    return tot / max(len(truth_dict), 1)

def check_and_save_submission(preds_dict, filename="submission.csv"):
    sp_ident = dict(zip(train["identity"], train["species"]))
    sp_test = dict(zip(test["id"], test["species"]))
    rows = []
    for img_id in test["id"]:
        labs = []
        for l in preds_dict.get(img_id, []):
            if l not in labs and (l == NEW or sp_ident.get(l) == sp_test[img_id]):
                labs.append(l)
        if not labs:
            labs = [NEW]
        rows.append((img_id, " ".join(labs[:5])))
    sub = pd.DataFrame(rows, columns=["id", "predictions"])
    assert len(sub) == len(test) and set(sub["id"]) == set(test["id"])
    assert not sub["id"].duplicated().any()
    out_path = OUT_DIR / filename
    sub.to_csv(out_path, index=False)
    print(f"Saved valid submission -> {out_path} ({len(sub)} rows)")
    return sub

print(f"Train: {len(train)} images, {train['identity'].nunique()} identities | Test: {len(test)} images")
print("Species breakdown (train):", train["species"].value_counts().to_dict())
print("Species breakdown (test): ", test["species"].value_counts().to_dict())

### 1.1 Power-Law Whitening & Encounter-Aware Disjoint 5-Fold Validation

* **Why Power-Whitening ($\alpha = 0.5$)?** Full PCA whitening (`whiten=True`, $\alpha=1.0$) divides by $\sqrt{\lambda_i}$, which amplifies tail noise in components 200–512. Power-whitening divides by $(\lambda_i + \epsilon)^{\alpha/2}$ with $\alpha=0.5$, dampening burst/background dominance while keeping discriminative fine-grained features clean.
* **Why Encounter-Aware Disjoint 5-Fold CV?** Each identity's near-duplicate burst photos (connected components at cosine > 0.80) are held out *together*, and each identity is held out as `new_individual` in **exactly one** of the 5 folds. This prevents duplicate-query inflation across folds and mirrors the true hidden test set.

In [ ]:
def compute_power_whitened_pca(F, row_map, sp_map, n_comp=512, alpha=0.5, eps=1e-4):
    """Per-species centered PCA with eigenvalue power-whitening: z / (var + eps)^(alpha/2)."""
    ids = sorted(row_map, key=row_map.get)
    spa = np.array([sp_map[i] for i in ids])
    E = np.zeros((len(F), n_comp), dtype=np.float32)
    for s in SPECIES_LIST:
        m = (spa == s)
        Z = F[m].astype(np.float64)
        Z = Z - Z.mean(axis=0, keepdims=True)
        k = min(n_comp, Z.shape[0] - 1, Z.shape[1])
        pca = PCA(n_components=k, svd_solver="randomized", random_state=SEED)
        U = pca.fit_transform(Z)
        scale = np.power(pca.explained_variance_ + eps, -0.5 * alpha)
        U = U * scale
        E[m, :k] = U.astype(np.float32)
    E /= (np.linalg.norm(E, axis=1, keepdims=True) + 1e-12)
    return E

def build_encounters(train_df, W, row_map, tau=0.80):
    """Connected components of same-identity photos with power-whitened cosine > tau."""
    enc = {}
    for ident, g in train_df.groupby("identity"):
        ids = g["id"].tolist()
        if len(ids) == 1:
            enc[ids[0]] = f"{ident}#0"
            continue
        X = W[[row_map[i] for i in ids]]
        adj = csr_matrix((X @ X.T) > tau)
        _, comp = connected_components(adj, directed=False)
        for img_id, c in zip(ids, comp):
            enc[img_id] = f"{ident}#{c}"
    return enc

def make_disjoint_folds(train_df, enc_map, n_folds=5, cap=10, seed=SEED):
    """
    Strictly disjoint 5-fold open-set validation:
    - Every identity is hidden as `new_individual` in exactly 1 fold (~20% unknown rate).
    - Known identities with >=2 encounters have disjoint encounters assigned across the remaining folds.
    """
    rng = np.random.default_rng(seed)
    lab = dict(zip(train_df["id"], train_df["identity"]))
    unk_fold = {}
    for _, d in train_df.groupby("species"):
        idents = rng.permutation(d["identity"].unique())
        for idx, ident in enumerate(idents):
            unk_fold[ident] = idx % n_folds

    # Pre-assign known encounters to folds != unk_fold[ident]
    enc_fold = {}
    for ident, g in train_df.groupby("identity"):
        uf = unk_fold[ident]
        encs = rng.permutation(sorted({enc_map[i] for i in g["id"]}))
        if len(encs) >= 2:
            avail_folds = [f for f in range(n_folds) if f != uf]
            # Leave at least 1 encounter always in gallery when that fold is queried
            for idx, e in enumerate(encs):
                enc_fold[e] = avail_folds[idx % len(avail_folds)]

    folds = []
    for f in range(n_folds):
        queries, hidden = [], set()
        for ident, g in train_df.groupby("identity"):
            ids = g["id"].to_numpy()
            if unk_fold[ident] == f:
                hidden.update(ids)
                q_pick = rng.choice(ids, min(cap, len(ids)), replace=False)
                queries.extend(q_pick.tolist())
            else:
                qi = [i for i in ids if enc_fold.get(enc_map[i], -1) == f]
                # Ensure at least 1 photo remains in gallery for this known identity
                if 0 < len(qi) < len(ids):
                    hidden.update(qi)
                    q_pick = rng.choice(qi, min(cap, len(qi)), replace=False)
                    queries.extend(q_pick.tolist())
        qset = set(queries)
        gallery = [i for i in train_df["id"] if i not in qset and i not in hidden]
        gal_idents = {lab[i] for i in gallery}
        truth = {q: (lab[q] if lab[q] in gal_idents else NEW) for q in sorted(qset)}
        folds.append({"fold": f, "queryIds": sorted(qset), "galleryIds": gallery, "truth": truth})
    return folds

W_PWR = compute_power_whitened_pca(F_RAW, ROW_MAP, SP_MAP, n_comp=512, alpha=0.50)
W_FULL = compute_power_whitened_pca(F_RAW, ROW_MAP, SP_MAP, n_comp=512, alpha=0.85)
ENC_MAP = build_encounters(train, W_PWR, ROW_MAP, tau=0.80)
FOLDS = make_disjoint_folds(train, ENC_MAP, n_folds=5)

print(f"Found {len(set(ENC_MAP.values()))} encounters across {len(train)} train photos.")
for f in FOLDS:
    t = pd.Series(f["truth"])
    print(f"  Fold {f['fold']}: {len(t):4d} queries | new={np.mean(t == NEW):.1%} | gallery={len(f['galleryIds']):4d}")

### 1.2 Multi-Space Retrieval, Hubness Reduction (CSLS) & k-Reciprocal Jaccard Reranking

For each fold (strictly fitting LDA **only on that fold's gallery**) and for the full test set:
1. **Regularized LDA** ($d=128$, `shrinkage='auto'`) fitted on `W_PWR` using gallery identities with $\ge 2$ photos (excluding singletons prevents zero-variance covariance distortion), blended 50/50 with `W_PWR`.
2. **CSLS (Cross-Domain Similarity Local Scaling)**: $\text{CSLS}(q, g) = 2\cos(q,g) - \bar{s}_{10}(q) - \bar{s}_{10}(g)$, where $\bar{s}_{10}(g)$ is gallery image $g$'s average cosine to its top-10 other-identity neighbors (hubness penalty!).
3. **k-Reciprocal Jaccard Similarity**: Measures mutual neighborhood overlap between query $q$ and gallery image $g$ inside the species' combined similarity graph — a classical Re-ID reranking signal that is 100% k-NN / distance-based.
4. **Identity Centroids & Viewpoint-Aware Stats**: Also computes cosine to each identity's L2-normalized mean vector (`CentroidCos`, `SameSideCentroidCos`) and query-normalized Z-scores (`CosZ`).

In [ ]:
TOP_ID = 35
RAW_SLOTS = 7
TOP_IMG_BASE = 2
TOP_IMG_DEEP_K = 5
TOP_IMG_DEEP_CAP = 8

def fit_gallery_lda_blend(W, row_map, sp_map, lab_map, gallery_ids, n_comp=128, mix=0.50):
    ids = sorted(row_map, key=row_map.get)
    spa = np.array([sp_map[i] for i in ids])
    L = np.zeros((len(W), n_comp), dtype=np.float32)
    for s in SPECIES_LIST:
        g = [i for i in gallery_ids if sp_map[i] == s]
        y = np.array([lab_map[i] for i in g])
        # Use identities with >= 2 images for cleaner within-class scatter estimation
        vc = pd.Series(y).value_counts()
        valid_ids = set(vc[vc >= 2].index)
        g_fit = [i for i in g if lab_map[i] in valid_ids]
        if len(valid_ids) < 3:
            g_fit = g
        y_fit = np.array([lab_map[i] for i in g_fit])
        k = min(n_comp, len(np.unique(y_fit)) - 1, W.shape[1])
        lda = LinearDiscriminantAnalysis(n_components=k, solver="eigen", shrinkage="auto")
        lda.fit(W[[row_map[i] for i in g_fit]], y_fit)
        L[spa == s, :k] = lda.transform(W[spa == s]).astype(np.float32)
    L /= (np.linalg.norm(L, axis=1, keepdims=True) + 1e-12)
    E = np.hstack([np.sqrt(1.0 - mix) * W, np.sqrt(mix) * L]).astype(np.float32)
    E /= (np.linalg.norm(E, axis=1, keepdims=True) + 1e-12)
    return E

def compute_k_reciprocal_jaccard(E_q, E_g, k1=15, k2=5):
    """
    Fast sparse k-reciprocal Jaccard similarity between queries (nq) and gallery (ng).
    Pure k-NN distance-based reranking (Zhong et al., CVPR 2017).
    """
    X = np.vstack([E_q, E_g]).astype(np.float32)
    N = len(X)
    nq = len(E_q)
    S = X @ X.T
    # Top-k1 neighbors for every node
    k1_eff = min(k1, N - 1)
    top_idx = np.argpartition(-S, k1_eff, axis=1)[:, :k1_eff + 1]
    # Build boolean membership matrix for fast reciprocal check
    rows = np.repeat(np.arange(N), k1_eff + 1)
    cols = top_idx.ravel()
    vals = np.exp(np.clip(S[rows, cols] * 5.0, -20, 20)).astype(np.float32)
    A = csr_matrix((vals, (rows, cols)), shape=(N, N))
    # Mutual (reciprocal) weights: elementwise min of A and A^T
    R = A.minimum(A.T).tocsr()
    # Local query expansion (average over top-k2 neighbors)
    k2_eff = min(k2, N - 1)
    top_k2 = np.argpartition(-S, k2_eff, axis=1)[:, :k2_eff + 1]
    r2 = np.repeat(np.arange(N), k2_eff + 1)
    c2 = top_k2.ravel()
    Qexp = csr_matrix((np.full(len(r2), 1.0 / (k2_eff + 1), dtype=np.float32), (r2, c2)), shape=(N, N))
    V = (Qexp @ R).tocsr()
    # L1-normalize rows so generalized Jaccard = sum(min(v_q, v_g)) / (2 - sum(min))
    # Fast cosine approximation on L2-normalized reciprocal feature vectors
    row_norms = np.sqrt(np.asarray(V.multiply(V).sum(axis=1)).ravel()) + 1e-12
    V = V.multiply(1.0 / row_norms[:, None]).tocsr()
    J = (V[:nq] @ V[nq:].T).toarray().astype(np.float32)
    return J

def retrieve_candidates(E, W_pwr, W_full, F_unit, row_map, sp_map, vp_map, query_ids, gallery_df):
    out_rows = []
    for s, G in gallery_df.groupby("species"):
        qids = [q for q in query_ids if sp_map[q] == s]
        if not qids:
            continue
        G = G.sort_values("identity").reset_index(drop=True)
        gid = G["id"].to_numpy()
        glab = G["identity"].to_numpy()
        gside = np.array([get_side(vp_map[g]) for g in gid])
        starts = np.r_[0, np.flatnonzero(glab[1:] != glab[:-1]) + 1]
        counts = np.diff(np.r_[starts, len(glab)])
        seg = np.repeat(np.arange(len(starts)), counts)
        labs = glab[starts]
        n_id = len(starts)

        q_idx = [row_map[q] for q in qids]
        g_idx = [row_map[g] for g in gid]

        Eq, Eg = E[q_idx], E[g_idx]
        Wq, Wg = W_pwr[q_idx], W_pwr[g_idx]
        WFq, WFg = W_full[q_idx], W_full[g_idx]
        Rq, Rg = F_unit[q_idx], F_unit[g_idx]

        # Identity centroids in main blended space E
        centroids = np.zeros((n_id, Eg.shape[1]), dtype=np.float32)
        for j in range(n_id):
            c_vec = Eg[starts[j]:starts[j] + counts[j]].mean(axis=0)
            centroids[j] = c_vec / (np.linalg.norm(c_vec) + 1e-12)
        S_cent = Eq @ centroids.T  # (nq, n_id)

        # Gallery-side hubness: mean top-10 similarity of each gallery image to OTHER identities
        S_gg = Eg @ Eg.T
        same_id_mask = (seg[:, None] == seg[None, :])
        S_gg[same_id_mask] = -1.0
        k_hub = min(10, S_gg.shape[1] - 1)
        g_hub = np.partition(S_gg, -k_hub, axis=1)[:, -k_hub:].mean(axis=1)  # (ng,)

        # Main similarity matrix + CSLS + k-Reciprocal Jaccard
        S = Eq @ Eg.T  # (nq, ng)
        q_hub = np.partition(S, -k_hub, axis=1)[:, -k_hub:].mean(axis=1)     # (nq,)
        S_csls = 2.0 * S - q_hub[:, None] - g_hub[None, :]
        J_mat = compute_k_reciprocal_jaccard(Eq, Eg, k1=15, k2=5)

        # Query-level background mean/std for Z-score normalization
        q_mean = S.mean(axis=1)
        q_std = S.std(axis=1) + 1e-6

        # Per-identity maxima in each space
        M_main = np.maximum.reduceat(S, starts, axis=1)
        M_csls = np.maximum.reduceat(S_csls, starts, axis=1)
        M_jac = np.maximum.reduceat(J_mat, starts, axis=1)
        M_raw = np.maximum.reduceat(Rq @ Rg.T, starts, axis=1)
        M_wp = np.maximum.reduceat(Wq @ Wg.T, starts, axis=1)
        M_wf = np.maximum.reduceat(WFq @ WFg.T, starts, axis=1)

        R_raw = np.argsort(np.argsort(-M_raw, axis=1), axis=1)
        R_wp = np.argsort(np.argsort(-M_wp, axis=1), axis=1)
        R_csls = np.argsort(np.argsort(-M_csls, axis=1), axis=1)
        R_jac = np.argsort(np.argsort(-M_jac, axis=1), axis=1)

        # Combined retrieval score for candidate selection (CSLS + Jaccard)
        M_sel = M_csls + 0.35 * M_jac

        for r, q in enumerate(qids):
            main_ord = np.argsort(-M_sel[r])
            order = main_ord[:TOP_ID - RAW_SLOTS]
            chosen = set(order.tolist())
            raw_extra = [j for j in np.argsort(-M_raw[r]) if j not in chosen][:RAW_SLOTS]
            order = np.r_[order, raw_extra].astype(int)

            top15_g = np.argsort(-S_csls[r])[:15]
            knn15 = np.bincount(seg[top15_g], minlength=n_id)
            qs = get_side(vp_map[q])

            top1_cos = float(M_main[r].max())
            top2_cos = float(np.partition(M_main[r], -2)[-2]) if n_id > 1 else -1.0
            top1_csls = float(M_csls[r].max())
            top2_csls = float(np.partition(M_csls[r], -2)[-2]) if n_id > 1 else -1.0
            top1_jac = float(M_jac[r].max())
            top2_jac = float(np.partition(M_jac[r], -2)[-2]) if n_id > 1 else 0.0

            for rk, j in enumerate(order):
                a, n = starts[j], counts[j]
                v_cos = S[r, a:a + n]
                v_csls = S_csls[r, a:a + n]
                v_jac = J_mat[r, a:a + n]
                v_hub = g_hub[a:a + n]
                o = np.argsort(-v_csls)

                cap_img = TOP_IMG_DEEP_CAP if rk < TOP_IMG_DEEP_K else TOP_IMG_BASE
                best_imgs = list(gid[a + o[:cap_img]])

                gs = gside[a:a + n]
                same_idx = np.flatnonzero(gs == qs) if qs in "LR" else np.array([], dtype=int)
                opp_idx = np.flatnonzero(gs == {"L": "R", "R": "L"}.get(qs, "-"))
                oth_idx = np.flatnonzero(gs == "O")

                if len(same_idx):
                    bs = gid[a + same_idx[np.argmax(v_csls[same_idx])]]
                    if bs not in best_imgs:
                        best_imgs.append(bs)

                same_max = float(v_cos[same_idx].max()) if len(same_idx) else np.nan
                same_csls = float(v_csls[same_idx].max()) if len(same_idx) else np.nan
                opp_max = float(v_cos[opp_idx].max()) if len(opp_idx) else np.nan
                oth_max = float(v_cos[oth_idx].max()) if len(oth_idx) else np.nan
                n_sides = len(set(gs) - {"U"})

                if qs in "LR":
                    n_comp = int((gs == qs).sum())
                elif qs == "O":
                    n_comp = int(sum(vp_map[g] == vp_map[q] for g in gid[a:a + n]))
                else:
                    n_comp = -1

                out_rows.append((
                    q, labs[j],
                    float(v_cos[o[0]]),
                    float(v_cos[o[1]]) if n > 1 else np.nan,
                    float(v_cos[o[:3]].mean()),
                    float(v_csls[o[0]]),
                    float(v_csls[o[1]]) if n > 1 else np.nan,
                    float(v_csls[o[:3]].mean()),
                    float(v_jac[o[0]]),
                    float(S_cent[r, j]),
                    float((v_cos[o[0]] - q_mean[r]) / q_std[r]),
                    float(v_hub[o[0]]),
                    float(q_hub[r]),
                    int(knn15[j]),
                    same_max, same_csls, opp_max, oth_max, n_sides,
                    rk, int(R_csls[r, j]), int(R_jac[r, j]),
                    top1_cos, top1_cos - top2_cos,
                    top1_csls, top1_csls - top2_csls,
                    top1_jac, top1_jac - top2_jac,
                    int(n), n_comp,
                    float(M_raw[r, j]), int(R_raw[r, j]),
                    float(M_wp[r, j]), int(R_wp[r, j]),
                    float(M_wf[r, j]),
                    best_imgs,
                ))

    cols = [
        "Query", "Identity",
        "CosMax", "Cos2nd", "CosMean3",
        "CSLSMax", "CSLS2nd", "CSLSMean3",
        "JacMax", "CentroidCos", "CosZ", "GalHub", "QHub",
        "Knn15", "SameSideMax", "SameSideCSLS", "OppSideMax", "OtherViewMax", "NSides",
        "IdRank", "CSLSRank", "JacRank",
        "QTop1Cos", "QGap12Cos", "QTop1CSLS", "QGap12CSLS", "QTop1Jac", "QGap12Jac",
        "NGal", "NCompGal",
        "RawCosMax", "RawRank", "WPCosMax", "WPRank", "WFCosMax",
        "BestImgs",
    ]
    return pd.DataFrame(out_rows, columns=cols)

t0 = time.time()
tr_sub = train[["id", "identity", "species"]]
FOLD_EMBS = {}
CAND_FOLDS = []

for f in FOLDS:
    E_f = fit_gallery_lda_blend(W_PWR, ROW_MAP, SP_MAP, LAB_MAP, f["galleryIds"])
    FOLD_EMBS[f["fold"]] = E_f
    gal_df = tr_sub[tr_sub["id"].isin(set(f["galleryIds"]))]
    cf = retrieve_candidates(E_f, W_PWR, W_FULL, F_UNIT, ROW_MAP, SP_MAP, VP_MAP, f["queryIds"], gal_df)
    cf["Fold"] = f["fold"]
    cf["QKey"] = f"{f['fold']}:" + cf["Query"]
    cf["Truth"] = cf["Query"].map(f["truth"])
    CAND_FOLDS.append(cf)

E_TEST = fit_gallery_lda_blend(W_PWR, ROW_MAP, SP_MAP, LAB_MAP, train["id"].tolist())
CAND_TEST = retrieve_candidates(E_TEST, W_PWR, W_FULL, F_UNIT, ROW_MAP, SP_MAP, VP_MAP, test["id"].tolist(), tr_sub)
CAND_TEST["QKey"] = CAND_TEST["Query"]

V_ALL = pd.concat(CAND_FOLDS, ignore_index=True)

# Diagnostic: Candidate Recall@35 and Raw Retrieval Baseline MAP@5
recalls = []
for f, cf in zip(FOLDS, CAND_FOLDS):
    c_map = cf.groupby("Query")["Identity"].apply(set).to_dict()
    known_q = [q for q, t in f["truth"].items() if t != NEW]
    recalls.append(np.mean([f["truth"][q] in c_map.get(q, set()) for q in known_q]))
print(f"Retrieval complete in {time.time() - t0:.1f}s | Mean Recall@{TOP_ID} (known): {np.mean(recalls):.4f}")

### 1.3 Fast Geometric SIFT Verification (5 Rich Features + Affine Sanity Check)

Improvements over the previous `P3Sift.py`:
* **3–4× Faster**: Pre-computes `float16` RootSIFT once per unique image (instead of re-computing per pair inside the thread pool) and focuses SIFT budget on the highest-leverage candidate pairs + query-to-query graph edges.
* **5 Geometric Features per Pair**: `(nGood, nInliers, inlierRatio, meanInlierSim, spatialSpread)`.
* **Curved-Flank Affine + Degeneracy Guard**: Uses `cv2.estimateAffine2D` (handles 3D body curvature/foreshortening), verifies `det > 0` and reasonable scale/shear, and falls back to `estimateAffinePartial2D` if degenerate.
* **Auto-Cached**: Saved to `Work/sift_rich_cache.pkl` so subsequent runs load instantly.

In [ ]:
SIFT_RATIO = 0.80
RANSAC_PX = 14.0
Q_NBR_K = 10

def get_query_neighbors(E, row_map, qids, sp_map, k=Q_NBR_K):
    """Top-k most similar other queries of the same species (NO test-set splitting!)."""
    out = {}
    for s in SPECIES_LIST:
        qs = [q for q in qids if sp_map[q] == s]
        if len(qs) <= 1:
            continue
        X = E[[row_map[q] for q in qs]]
        S = X @ X.T
        np.fill_diagonal(S, -9.0)
        kk = min(k, len(qs) - 1)
        idx = np.argpartition(-S, kk, axis=1)[:, :kk]
        for a, q in enumerate(qs):
            nbrs = idx[a]
            nbrs = nbrs[np.argsort(-S[a, nbrs])]
            out[q] = [(qs[b], float(S[a, b])) for b in nbrs]
    return out

FOLD_QNBRS = {f["fold"]: get_query_neighbors(FOLD_EMBS[f["fold"]], ROW_MAP, f["queryIds"], SP_MAP) for f in FOLDS}
TEST_QNBRS = get_query_neighbors(E_TEST, ROW_MAP, test["id"].tolist(), SP_MAP)

# Collect all required SIFT pairs: candidate pairs + query-query neighbor pairs (top 6 neighbors)
req_pairs = set()
for df_c in [V_ALL, CAND_TEST]:
    # Only SIFT-verify top-25 candidates per query to stay well under 10 min runtime
    sub_c = df_c[df_c["IdRank"] < 25]
    for q, imgs in zip(sub_c["Query"], sub_c["BestImgs"]):
        for g in imgs:
            req_pairs.add((q, g))

for nb_dict in list(FOLD_QNBRS.values()) + [TEST_QNBRS]:
    for q, lst in nb_dict.items():
        for q2, sim in lst[:6]:
            if sim > 0.35:
                req_pairs.add((q, q2) if q < q2 else (q2, q))

print(f"Total unique SIFT pairs requested: {len(req_pairs):,}")

SIFT_CACHE_PATH = WORK_DIR / "sift_rich_cache.pkl"
NKP_CACHE_PATH = WORK_DIR / "nkp_cache.pkl"

if SIFT_CACHE_PATH.exists() and NKP_CACHE_PATH.exists():
    with open(SIFT_CACHE_PATH, "rb") as fh:
        SIFT_CACHE = pickle.load(fh)
    with open(NKP_CACHE_PATH, "rb") as fh:
        NKP_MAP = pickle.load(fh)
else:
    SIFT_CACHE, NKP_MAP = {}, {}

todo_pairs = [p for p in req_pairs if p not in SIFT_CACHE]
print(f"Cached: {len(SIFT_CACHE):,} | To compute: {len(todo_pairs):,}")

def load_sift_store(img_ids):
    """Loads SIFT xy (float32) and RootSIFT desc (float16) for requested images."""
    train_set = set(train["id"])
    npz_handles = {}
    empty = (np.zeros((0, 2), np.float32), np.zeros((0, 128), np.float16))
    store = {}
    for img_id in img_ids:
        src = SIFT_TRAIN_PATH if img_id in train_set else SIFT_TEST_PATH
        xy, d = None, None
        if src.is_dir():
            fx, fd = src / f"{img_id}_xy.npy", src / f"{img_id}_desc.npy"
            if fx.exists() and fd.exists():
                xy, d = np.load(fx), np.load(fd)
        else:
            if src not in npz_handles:
                npz_handles[src] = np.load(src)
            z = npz_handles[src]
            kx, kd = f"{img_id}_xy", f"{img_id}_desc"
            if kx in z.files and kd in z.files:
                xy, d = z[kx], z[kd]
        if xy is None or len(d) == 0:
            store[img_id] = empty
        else:
            d = d.astype(np.float32).reshape(-1, 128)
            d /= (d.sum(axis=1, keepdims=True) + 1e-7)
            d = np.sqrt(d).astype(np.float16)
            store[img_id] = (xy.astype(np.float32).reshape(-1, 2), d)
    for h in npz_handles.values():
        h.close()
    return store

def match_pair_rich(qxy, qd32, gxy, gd16):
    """
    Returns (nGood, nInliers, inlierRatio, meanInlierSim, spatialSpread).
    """
    if len(qd32) < 2 or len(gd16) < 2:
        return (0, 0, 0.0, 0.0, 0.0)
    gd32 = gd16.astype(np.float32)
    S = qd32 @ gd32.T  # unit RootSIFT cosine
    r = np.arange(len(S))
    j1 = S.argmax(axis=1)
    s1 = S[r, j1].copy()
    S[r, j1] = -np.inf
    s2 = S.max(axis=1)
    d1 = np.sqrt(np.maximum(2.0 - 2.0 * s1, 0.0))
    d2 = np.sqrt(np.maximum(2.0 - 2.0 * s2, 0.0))
    qi = np.flatnonzero(d1 < SIFT_RATIO * d2)
    if len(qi) == 0:
        return (0, 0, 0.0, 0.0, 0.0)
    gj = j1[qi]
    # One-to-one deduplication (keep best match per gallery keypoint)
    o = np.argsort(d1[qi])
    qi, gj = qi[o], gj[o]
    _, u = np.unique(gj, return_index=True)
    qi, gj = qi[u], gj[u]
    n_good = int(len(qi))
    if n_good < 4:
        return (n_good, 0, 0.0, 0.0, 0.0)

    pts_q, pts_g = qxy[qi], gxy[gj]
    M, inl = cv2.estimateAffine2D(
        pts_q, pts_g, method=cv2.RANSAC,
        ransacReprojThreshold=RANSAC_PX, maxIters=1500, confidence=0.99
    )
    use_partial = True
    if M is not None and inl is not None:
        A = M[:, :2]
        det = float(np.linalg.det(A))
        sv = np.linalg.svd(A, compute_uv=False)
        if det > 0.05 and sv[1] > 0.2 and sv[0] < 4.5 and (sv[0] / (sv[1] + 1e-6)) < 3.5:
            use_partial = False

    if use_partial:
        _, inl = cv2.estimateAffinePartial2D(
            pts_q, pts_g, method=cv2.RANSAC,
            ransacReprojThreshold=RANSAC_PX, maxIters=1500, confidence=0.99
        )

    if inl is None:
        return (n_good, 0, 0.0, 0.0, 0.0)

    mask = inl.ravel().astype(bool)
    n_inl = int(mask.sum())
    if n_inl < 3:
        return (n_good, n_inl, float(n_inl / n_good), 0.0, 0.0)

    inlier_sims = s1[qi[mask]]
    mean_sim = float(inlier_sims.mean())
    # Spatial spread of inliers on query image (normalized by overall query keypoint spread)
    q_span = float(np.hypot(qxy[:, 0].std(), qxy[:, 1].std()) + 1e-5)
    inl_pts = pts_q[mask]
    inl_spread = float(np.hypot(inl_pts[:, 0].std(), inl_pts[:, 1].std()) / q_span)
    return (n_good, n_inl, float(n_inl / n_good), mean_sim, inl_spread)

if todo_pairs:
    t0 = time.time()
    need_ids = sorted({i for p in todo_pairs for i in p})
    sift_store = load_sift_store(need_ids)
    for img_id, (_, d) in sift_store.items():
        NKP_MAP[img_id] = len(d)
    print(f"Loaded & converted RootSIFT for {len(need_ids):,} images in {time.time() - t0:.1f}s")

    by_q = {}
    for q, g in todo_pairs:
        by_q.setdefault(q, []).append(g)

    def _worker(item):
        q, gs = item
        qxy, qd16 = sift_store[q]
        qd32 = qd16.astype(np.float32)
        return [((q, g), match_pair_rich(qxy, qd32, sift_store[g][0], sift_store[g][1])) for g in gs]

    t1, done = time.time(), 0
    with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
        for res in ex.map(_worker, by_q.items(), chunksize=12):
            SIFT_CACHE.update(res)
            done += len(res)
            if done % 40000 < len(res):
                rate = done / max(time.time() - t1, 1e-3)
                print(f"  SIFT progress: {done:,}/{len(todo_pairs):,} ({rate:.0f} pairs/s)")

    with open(SIFT_CACHE_PATH, "wb") as fh:
        pickle.dump(SIFT_CACHE, fh)
    with open(NKP_CACHE_PATH, "wb") as fh:
        pickle.dump(NKP_MAP, fh)
    del sift_store
    gc.collect()
    print(f"SIFT matching finished in {time.time() - t1:.1f}s -> cached to {SIFT_CACHE_PATH}")

### 1.4 Feature Engineering (Multi-Space + Hubness + Rich Comparable-View SIFT)

Separates *"matched SIFT on a comparable view and failed"* from *"never had a comparable-view photo in gallery"*, and adds query-relative margins across CSLS, Jaccard, Raw cosine, and geometric SIFT inliers/spread.

In [ ]:
def second_largest(s):
    s = s.dropna()
    return s.nlargest(2).iloc[-1] if len(s) >= 2 else np.nan

def add_rich_features(df, sift_cache, nkp_map, vp_map, sp_map):
    c = df.copy()
    n = len(c)
    inl_max = np.full(n, np.nan, dtype=np.float32)
    good_max = np.full(n, np.nan, dtype=np.float32)
    inl_sum = np.full(n, np.nan, dtype=np.float32)
    inl_cnt8 = np.full(n, np.nan, dtype=np.float32)
    ratio_max = np.full(n, np.nan, dtype=np.float32)
    sim_max = np.full(n, np.nan, dtype=np.float32)
    spread_max = np.full(n, np.nan, dtype=np.float32)
    nkp_g = np.full(n, np.nan, dtype=np.float32)

    n_comp_chk = np.full(n, np.nan, dtype=np.float32)
    comp_inl_max = np.full(n, np.nan, dtype=np.float32)
    comp_inl_norm = np.full(n, np.nan, dtype=np.float32)
    comp_spread = np.full(n, np.nan, dtype=np.float32)
    noncomp_inl = np.full(n, np.nan, dtype=np.float32)
    vrel = np.zeros(n, dtype=np.int8)

    queries = c["Query"].tolist()
    best_imgs_list = c["BestImgs"].tolist()

    for i, (q, imgs) in enumerate(zip(queries, best_imgs_list)):
        vrel[i] = view_relation(vp_map[q], vp_map[imgs[0]])
        nkp_g[i] = nkp_map.get(imgs[0], np.nan)
        qs = get_side(vp_map[q])
        nkq_i = nkp_map.get(q, 0)

        vals = []
        comp_vals, noncomp_vals = [], []
        for g in imgs:
            res = sift_cache.get((q, g))
            if res is None:
                continue
            vals.append(res)
            same = (get_side(vp_map[g]) == qs) if qs in "LR" else (vp_map[g] == vp_map[q] if qs == "O" else False)
            if same:
                comp_vals.append((res[1], res[1] / (min(nkq_i, nkp_map.get(g, 0)) + 1.0), res[4]))
            else:
                noncomp_vals.append(res[1])

        if vals:
            inl_max[i] = max(v[1] for v in vals)
            good_max[i] = max(v[0] for v in vals)
            inl_sum[i] = sum(v[1] for v in vals)
            inl_cnt8[i] = sum(v[1] >= 8 for v in vals)
            # Take ratio/sim/spread from the best-inlier image
            best_v = max(vals, key=lambda v: (v[1], v[3]))
            ratio_max[i] = best_v[2]
            sim_max[i] = best_v[3]
            spread_max[i] = best_v[4]

        if qs != "U":
            n_comp_chk[i] = len(comp_vals)
        if comp_vals:
            best_cv = max(comp_vals, key=lambda x: x[0])
            comp_inl_max[i] = best_cv[0]
            comp_inl_norm[i] = best_cv[1]
            comp_spread[i] = best_cv[2]
        if noncomp_vals:
            noncomp_inl[i] = max(noncomp_vals)

    c["InlMax"] = inl_max
    c["GoodMax"] = good_max
    c["InlSum"] = inl_sum
    c["InlCnt8"] = inl_cnt8
    c["InlRatio"] = ratio_max
    c["InlMeanSim"] = sim_max
    c["InlSpread"] = spread_max
    c["InlQuality"] = np.log1p(np.nan_to_num(inl_max, nan=0.0)) * np.nan_to_num(spread_max, nan=0.0)
    c["NChecked"] = c["BestImgs"].str.len()

    c["NCompChecked"] = n_comp_chk
    c["CompInlMax"] = comp_inl_max
    c["CompInlNorm"] = comp_inl_norm
    c["CompSpread"] = comp_spread
    c["NonCompInlMax"] = noncomp_inl
    c["NoCompView"] = np.where(c["NCompGal"] < 0, np.nan, (c["NCompGal"] == 0).astype(np.float32))
    c["LogNCompGal"] = np.log1p(c["NCompGal"].clip(lower=0))

    # Query-relative aggregations
    g_inl = c.groupby("QKey")["InlMax"]
    c["QInlMax"] = g_inl.transform("max")
    c["QInl2nd"] = g_inl.transform(second_largest)
    c["InlRank"] = g_inl.rank(ascending=False, method="min")
    c["InlFrac"] = c["InlMax"] / (c["QInlMax"] + 1.0)
    c["InlMargin"] = c["InlMax"] - np.where(c["InlMax"] >= c["QInlMax"], c["QInl2nd"], c["QInlMax"])
    c["QN12"] = (c["InlMax"] >= 12).groupby(c["QKey"]).transform("sum")
    c["QCompInlMax"] = c.groupby("QKey")["CompInlMax"].transform("max")

    nkq = c["Query"].map(nkp_map).astype(np.float32)
    c["InlNorm"] = c["InlMax"] / (np.minimum(nkq, nkp_g) + 1.0)
    c["LogNkpQ"] = np.log1p(nkq)
    c["LogNGal"] = np.log1p(c["NGal"])

    # Cosine / CSLS / Jaccard gaps
    c["CosGap"] = c["QTop1Cos"] - c["CosMax"]
    c["CSLSGap"] = c["QTop1CSLS"] - c["CSLSMax"]
    c["JacGap"] = c["QTop1Jac"] - c["JacMax"]
    c["SameGap"] = c["CosMax"] - c["SameSideMax"]
    c["OppGap"] = c["CosMax"] - c["OppSideMax"]
    c["HasSameSide"] = c["SameSideMax"].notna().astype(np.int8)

    c["QRawTop1"] = c.groupby("QKey")["RawCosMax"].transform("max")
    c["QWPTop1"] = c.groupby("QKey")["WPCosMax"].transform("max")
    c["RawGap"] = c["QRawTop1"] - c["RawCosMax"]
    c["WPGap"] = c["QWPTop1"] - c["WPCosMax"]
    c["QKnnTop"] = c.groupby("QKey")["Knn15"].transform("max")

    # Best-image raw cosine
    q_r = c["Query"].map(ROW_MAP).to_numpy()
    b_r = c["BestImgs"].str[0].map(ROW_MAP).to_numpy()
    c["RawCosBest"] = (F_UNIT[q_r] * F_UNIT[b_r]).sum(axis=1)

    c["VRel"] = vrel
    c["SpeciesC"] = c["Query"].map(lambda q: SPECIES_LIST.index(sp_map[q])).astype(np.int8)
    c["QSide"] = c["Query"].map(lambda q: SIDE_CODE[get_side(vp_map[q])]).astype(np.int8)
    return c

t0 = time.time()
V_FEAT = add_rich_features(V_ALL, SIFT_CACHE, NKP_MAP, VP_MAP, SP_MAP)
T_FEAT = add_rich_features(CAND_TEST, SIFT_CACHE, NKP_MAP, VP_MAP, SP_MAP)
print(f"Engineered features in {time.time() - t0:.1f}s | V_FEAT: {V_FEAT.shape} | T_FEAT: {T_FEAT.shape}")

### 1.5 Stage 1 Base Models & Stage 2 Full-Graph Transductive Propagation (`networkx`)

* **Stage 1**: Out-of-fold LightGBM predicts $P(\text{match})$ per candidate and $P(\text{new\_individual})$ per query (strictly grouped by `identity` so no animal leaks across train/val).
* **Stage 2 (Full Graph)**: Connects queries $(q_1, q_2)$ with strong visual or geometric SIFT evidence ($\text{cos} \ge 0.78$ or $\text{SIFT inliers} \ge 12$) using `networkx` — **without splitting the test set in half!** Computes both direct neighbor evidence and connected-component statistics (`CompMaxP`, `CompMeanNew`, `CompSize`).

In [ ]:
ID_FEATS_1 = [
    "CosMax", "Cos2nd", "CosMean3",
    "CSLSMax", "CSLS2nd", "CSLSMean3", "CSLSGap", "CSLSRank",
    "JacMax", "JacGap", "JacRank", "CentroidCos", "CosZ", "GalHub", "QHub",
    "Knn15", "SameSideMax", "SameSideCSLS", "SameGap", "OppSideMax", "OppGap",
    "OtherViewMax", "NSides", "HasSameSide", "IdRank",
    "QTop1Cos", "QGap12Cos", "QTop1CSLS", "QGap12CSLS", "QTop1Jac", "QGap12Jac",
    "CosGap", "LogNGal", "RawCosBest", "RawCosMax", "RawRank", "RawGap",
    "WPCosMax", "WPRank", "WPGap", "WFCosMax",
    "InlMax", "GoodMax", "InlSum", "InlCnt8", "InlRatio", "InlMeanSim", "InlSpread", "InlQuality",
    "NChecked", "InlRank", "QInlMax", "InlFrac", "InlNorm", "InlMargin", "QN12",
    "NCompChecked", "CompInlMax", "CompInlNorm", "CompSpread", "NonCompInlMax",
    "NoCompView", "LogNCompGal", "LogNkpQ", "VRel", "SpeciesC", "QSide",
]

Q_BASE_1 = [
    "QTop1Cos", "QGap12Cos", "QTop1CSLS", "QGap12CSLS", "QTop1Jac", "QGap12Jac",
    "QHub", "QInlMax", "QInl2nd", "QN12", "QKnnTop", "LogNkpQ", "SpeciesC", "QSide",
    "QRawTop1", "QWPTop1", "QCompInlMax",
]
Q_FEATS_1 = Q_BASE_1 + ["PMax", "P2nd", "PSum", "PGap12", "PEntropy"]

LGB_ID_PARAMS = dict(
    n_estimators=700, learning_rate=0.02, num_leaves=15, min_child_samples=80,
    subsample=0.8, subsample_freq=1, colsample_bytree=0.75, reg_alpha=1.0, reg_lambda=5.0,
    verbose=-1, random_state=SEED, n_jobs=N_WORKERS,
)
LGB_NEW_PARAMS = dict(
    n_estimators=500, learning_rate=0.02, num_leaves=15, min_child_samples=50,
    subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_alpha=1.0, reg_lambda=5.0,
    verbose=-1, random_state=SEED, n_jobs=N_WORKERS,
)

def build_query_table(df, base_cols):
    g = df.groupby("QKey", sort=False)
    Q = g[base_cols].first()
    Q["PMax"] = g["P"].max()
    Q["P2nd"] = g["P"].apply(second_largest).fillna(0.0)
    Q["PSum"] = g["P"].sum()
    Q["PGap12"] = Q["PMax"] - Q["P2nd"]
    # Normalized probability entropy across candidates
    def _ent(p):
        s = p.sum()
        if s <= 1e-9:
            return 0.0
        q = np.clip(p / s, 1e-9, 1.0)
        return float(-(q * np.log(q)).sum())
    Q["PEntropy"] = g["P"].apply(_ent)
    return Q

def oof_predict_lgb(X, y, groups, params, n_splits=5):
    out = np.zeros(len(X), dtype=np.float64)
    for tr, va in GroupKFold(n_splits).split(X, y, groups):
        m = lgb.LGBMClassifier(**params).fit(X.iloc[tr], y[tr])
        out[va] = m.predict_proba(X.iloc[va])[:, 1]
    return out

def eval_stage_map5(df, p_new_series, bias=0.0):
    preds = {}
    truth = {}
    for qk, g in df.groupby("QKey", sort=False):
        truth[qk] = g["Truth"].iloc[0]
        p0 = float(np.clip(p_new_series[qk], 1e-6, 1.0 - 1e-6))
        pn = float(expit(logit(p0) + bias))
        pk = g["P"].to_numpy()
        sk = (1.0 - pn) * (pk / max(pk.sum(), 1.0))
        labs = g["Identity"].tolist() + [NEW]
        scores = np.r_[sk, pn]
        preds[qk] = [labs[i] for i in np.argsort(-scores)[:5]]
    return map5_score(truth, preds)

# ---- Run Stage 1 ----
t0 = time.time()
y_id = (V_FEAT["Identity"] == V_FEAT["Truth"]).to_numpy(int)
id_groups = V_FEAT["Query"].map(LAB_MAP).to_numpy()

V_FEAT["P"] = oof_predict_lgb(V_FEAT[ID_FEATS_1], y_id, id_groups, LGB_ID_PARAMS)
m_id_1 = lgb.LGBMClassifier(**LGB_ID_PARAMS).fit(V_FEAT[ID_FEATS_1], y_id)
T_FEAT["P"] = m_id_1.predict_proba(T_FEAT[ID_FEATS_1])[:, 1]

QV1 = build_query_table(V_FEAT, Q_BASE_1)
QT1 = build_query_table(T_FEAT, Q_BASE_1)
y_new = (V_FEAT.groupby("QKey")["Truth"].first().loc[QV1.index] == NEW).to_numpy(int)
q_groups = QV1.index.str.split(":").str[1].map(LAB_MAP).to_numpy()

p_new_v1 = pd.Series(oof_predict_lgb(QV1[Q_FEATS_1], y_new, q_groups, LGB_NEW_PARAMS), index=QV1.index)
m_new_1 = lgb.LGBMClassifier(**LGB_NEW_PARAMS).fit(QV1[Q_FEATS_1], y_new)
p_new_t1 = pd.Series(m_new_1.predict_proba(QT1[Q_FEATS_1])[:, 1], index=QT1.index)

s1_map5 = eval_stage_map5(V_FEAT, p_new_v1, bias=0.0)
print(f"Stage 1 (No Graph) CV MAP@5: {s1_map5:.5f} ({time.time() - t0:.1f}s)")

# ---- Stage 2: Full-Graph Transductive Propagation via NetworkX ----
def add_graph_features(df, p_new_map, q_nbrs, prefix, sift_cache, cos_thr=0.78, inl_thr=12):
    """
    Builds a query-query graph per fold/test using absolute similarity/SIFT thresholds
    (density-consistent between CV and full Test — no test splitting!).
    """
    P_dict = {(q, ident): p for q, ident, p in zip(df["Query"], df["Identity"], df["P"])}
    q_list = df["Query"].unique().tolist()

    G = nx.Graph()
    G.add_nodes_from(q_list)
    links = {}
    q_cos, q_inl, q_nbr_new = {}, {}, {}

    for q in q_list:
        lk = []
        for q2, cs in q_nbrs.get(q, []):
            key = (q, q2) if q < q2 else (q2, q)
            s_res = sift_cache.get(key)
            inl = s_res[1] if s_res is not None else 0
            lk.append((q2, cs, inl))
            if cs >= cos_thr or inl >= inl_thr:
                w = max(cs, min(1.0, inl / 25.0))
                G.add_edge(q, q2, weight=w)
        links[q] = lk
        q_cos[q] = max((x[1] for x in lk), default=0.0)
        q_inl[q] = max((x[2] for x in lk), default=0)
        strongest = max(lk, key=lambda x: (x[2], x[1]), default=None)
        q_nbr_new[q] = p_new_map.get(prefix + strongest[0], np.nan) if strongest else np.nan

    # Connected-component level summaries (other nodes in same component)
    comp_members = {}
    comp_size = {}
    comp_mean_new = {}
    for comp in nx.connected_components(G):
        c_list = list(comp)
        sz = len(c_list)
        for q in c_list:
            others = [x for x in c_list if x != q]
            comp_members[q] = others
            comp_size[q] = sz
            comp_mean_new[q] = float(np.mean([p_new_map.get(prefix + x, np.nan) for x in others])) if others else np.nan

    out = df.copy()
    nbr_p, nbr_p_inl, comp_max_p = [], [], []
    for q, ident in zip(out["Query"], out["Identity"]):
        lk = links[q]
        nbr_p.append(max((cs * P_dict.get((q2, ident), 0.0) for q2, cs, _ in lk), default=0.0))
        nbr_p_inl.append(max((P_dict.get((q2, ident), 0.0) for q2, _, inl in lk if inl >= inl_thr), default=0.0))
        others = comp_members[q]
        comp_max_p.append(max((P_dict.get((q2, ident), 0.0) for q2 in others), default=0.0) if others else 0.0)

    out["NbrP"] = nbr_p
    out["NbrPInl"] = nbr_p_inl
    out["CompMaxP"] = comp_max_p
    out["NbrPMax"] = out.groupby("QKey")["NbrP"].transform("max")
    out["CompMaxPTop"] = out.groupby("QKey")["CompMaxP"].transform("max")
    out["NbrCos"] = out["Query"].map(q_cos)
    out["NbrInl"] = out["Query"].map(q_inl)
    out["NbrNew"] = out["Query"].map(q_nbr_new)
    out["CompSize"] = np.log1p(out["Query"].map(comp_size))
    out["CompMeanNew"] = out["Query"].map(comp_mean_new)
    return out

v_parts = []
for f in FOLDS:
    part = V_FEAT[V_FEAT["Fold"] == f["fold"]]
    v_parts.append(add_graph_features(part, p_new_v1.to_dict(), FOLD_QNBRS[f["fold"]], f"{f['fold']}:", SIFT_CACHE))
V2 = pd.concat(v_parts, ignore_index=True)
T2 = add_graph_features(T_FEAT, p_new_t1.to_dict(), TEST_QNBRS, "", SIFT_CACHE)

ID_FEATS_2 = ID_FEATS_1 + ["NbrP", "NbrPInl", "CompMaxP", "NbrCos", "NbrInl", "CompSize"]
Q_BASE_2 = Q_BASE_1 + ["NbrCos", "NbrInl", "NbrNew", "NbrPMax", "CompMaxPTop", "CompSize", "CompMeanNew"]
Q_FEATS_2 = Q_BASE_2 + ["PMax", "P2nd", "PSum", "PGap12", "PEntropy"]

y_id_2 = (V2["Identity"] == V2["Truth"]).to_numpy(int)
id_groups_2 = V2["Query"].map(LAB_MAP).to_numpy()

V2["P"] = oof_predict_lgb(V2[ID_FEATS_2], y_id_2, id_groups_2, LGB_ID_PARAMS)
m_id_2 = lgb.LGBMClassifier(**LGB_ID_PARAMS).fit(V2[ID_FEATS_2], y_id_2)
T2["P"] = m_id_2.predict_proba(T2[ID_FEATS_2])[:, 1]

QV2 = build_query_table(V2, Q_BASE_2)
QT2 = build_query_table(T2, Q_BASE_2)
y_new_2 = (V2.groupby("QKey")["Truth"].first().loc[QV2.index] == NEW).to_numpy(int)
q_groups_2 = QV2.index.str.split(":").str[1].map(LAB_MAP).to_numpy()

p_new_v2 = pd.Series(oof_predict_lgb(QV2[Q_FEATS_2], y_new_2, q_groups_2, LGB_NEW_PARAMS), index=QV2.index)
m_new_2 = lgb.LGBMClassifier(**LGB_NEW_PARAMS).fit(QV2[Q_FEATS_2], y_new_2)
p_new_t2 = pd.Series(m_new_2.predict_proba(QT2[Q_FEATS_2])[:, 1], index=QT2.index)

s2_map5 = eval_stage_map5(V2, p_new_v2, bias=0.0)
print(f"Stage 2 (+Full Graph) CV MAP@5: {s2_map5:.5f} (delta: {s2_map5 - s1_map5:+.5f})")

### 1.6 Diverse Multi-Family Ensemble + Convex Log-Loss Blending + Per-Species Prior-Shift EM

To prevent overfitting (a major risk when tuning 18 parameters on discrete MAP@5):
1. We train **5 genuinely diverse model families** out-of-fold (strictly grouped by animal `identity`):
   * `lgb_clf`: 2-stage LightGBM (calibrated candidate + query `new_individual` probabilities)
   * `cb_clf`: 2-stage CatBoost (oblivious symmetric trees — strong variance reduction)
   * `mlp_clf`: 2-stage Regularized MLP (smooth non-tree decision boundary)
   * `lgb_rank`: LightGBM LambdaRank (listwise ranking with dedicated `IsNewRow` row, 2 seeds)
   * `xgb_rank`: XGBoost `rank:ndcg` (depth-wise regularized listwise ranker)
2. **Smooth Convex Calibration**: Each ranker's logits are converted to per-query probabilities via temperature-scaled softmax $\text{softmax}(s / T + b \cdot \mathbb{I}_{\text{new}})$ fitted by minimizing **out-of-fold multiclass log-loss** (smooth, strictly convex, cannot overfit).
3. **Simplex Ensemble Weights**: Found by minimizing **out-of-fold log-loss** over the probability simplex ($\sum w_m = 1, w_m \ge 0$) — zero greedy coordinate ascent on MAP@5!
4. **Per-Species Prior-Shift EM (Saerens et al.)**: Estimates the hidden test `new_individual` rate separately for each species and applies the exact Bayes log-odds adjustment.

In [ ]:
def build_rank_table(df, q_cols):
    R = df.copy()
    R["IsNewRow"] = 0
    q_first = df.groupby("QKey", sort=False).first().reset_index()
    n_rows = pd.DataFrame({
        "QKey": q_first["QKey"],
        "Query": q_first["Query"],
        "Identity": NEW,
        "IsNewRow": 1,
        "SpeciesC": q_first["SpeciesC"],
    })
    if "Truth" in q_first.columns:
        n_rows["Truth"] = q_first["Truth"].values
    for col in q_cols:
        if col not in n_rows.columns:
            n_rows[col] = q_first[col].values
    R = pd.concat([R, n_rows], ignore_index=True)
    return R.sort_values(["QKey", "IsNewRow"], kind="stable").reset_index(drop=True)

RANK_FEATS = list(dict.fromkeys(ID_FEATS_2 + [c for c in Q_BASE_2 if c not in ID_FEATS_2] + ["IsNewRow"]))
RV = build_rank_table(V2, Q_BASE_2)
RT = build_rank_table(T2, Q_BASE_2)

y_rv = (RV["Identity"] == RV["Truth"]).to_numpy(int)
rv_groups = RV["Query"].map(LAB_MAP).to_numpy()
rv_qcode, rv_qkeys = pd.factorize(RV["QKey"])
rt_qcode, rt_qkeys = pd.factorize(RT["QKey"])
rv_is_new = (RV["IsNewRow"].to_numpy() == 1)
rt_is_new = (RT["IsNewRow"].to_numpy() == 1)

def clf_to_row_probs(df_cand, p_new_series, R_target):
    """Maps 2-stage (candidate P, query pNew) to normalized per-row probabilities on R_target."""
    p_map = df_cand.set_index(["QKey", "Identity"])["P"]
    code = pd.factorize(R_target["QKey"])[0]
    is_new = (R_target["IsNewRow"].to_numpy() == 1)
    p = pd.MultiIndex.from_frame(R_target[["QKey", "Identity"]]).map(p_map).to_numpy(dtype=np.float64)
    p = np.nan_to_num(p, nan=0.0)
    pn = np.clip(R_target["QKey"].map(p_new_series).to_numpy(dtype=np.float64), 1e-6, 1.0 - 1e-6)
    p_sum = pd.Series(p * (~is_new)).groupby(code).transform("sum").to_numpy()
    raw_p = np.where(is_new, pn, (1.0 - pn) * (p / np.maximum(p_sum, 1.0)))
    tot = pd.Series(raw_p).groupby(code).transform("sum").to_numpy()
    return np.clip(raw_p / np.maximum(tot, 1e-12), 1e-9, 1.0)

def fit_two_stage_clf(name, model_fn):
    t0 = time.time()
    vc, tc = V2.copy(), T2.copy()
    oof_p = np.zeros(len(vc), dtype=np.float64)
    for tr, va in GroupKFold(5).split(vc, y_id_2, id_groups_2):
        m = model_fn().fit(vc[ID_FEATS_2].iloc[tr], y_id_2[tr])
        oof_p[va] = m.predict_proba(vc[ID_FEATS_2].iloc[va])[:, 1]
    vc["P"] = oof_p
    tc["P"] = model_fn().fit(vc[ID_FEATS_2], y_id_2).predict_proba(tc[ID_FEATS_2])[:, 1]

    qv, qt = build_query_table(vc, Q_BASE_2), build_query_table(tc, Q_BASE_2)
    oof_n = np.zeros(len(qv), dtype=np.float64)
    for tr, va in GroupKFold(5).split(qv, y_new_2, q_groups_2):
        m = model_fn().fit(qv[Q_FEATS_2].iloc[tr], y_new_2[tr])
        oof_n[va] = m.predict_proba(qv[Q_FEATS_2].iloc[va])[:, 1]
    pn_v = pd.Series(oof_n, index=qv.index)
    pn_t = pd.Series(model_fn().fit(qv[Q_FEATS_2], y_new_2).predict_proba(qt[Q_FEATS_2])[:, 1], index=qt.index)
    print(f"  Trained {name:10s} in {time.time() - t0:.1f}s")
    return clf_to_row_probs(vc, pn_v, RV), clf_to_row_probs(tc, pn_t, RT)

def softmax_by_query(scores, qcode, is_new, temp=1.0, new_shift=0.0):
    s = scores / max(temp, 1e-3) + new_shift * is_new
    s = s - pd.Series(s).groupby(qcode).transform("max").to_numpy()
    e = np.exp(np.clip(s, -40, 0))
    denom = pd.Series(e).groupby(qcode).transform("sum").to_numpy()
    return np.clip(e / np.maximum(denom, 1e-12), 1e-9, 1.0)

def calibrate_ranker_probs(oof_scores, test_scores):
    """Fits 2 scalars (temperature, new_shift) by minimizing smooth OOF multiclass log-loss."""
    pos_mask = (y_rv == 1)
    def loss_fn(params):
        temp, shift = params
        p = softmax_by_query(oof_scores, rv_qcode, rv_is_new, temp=np.exp(temp), new_shift=shift)
        return -float(np.mean(np.log(p[pos_mask])))
    res = minimize(loss_fn, x0=[0.0, 0.0], method="Nelder-Mead")
    t_opt, b_opt = float(np.exp(res.x[0])), float(res.x[1])
    return (
        softmax_by_query(oof_scores, rv_qcode, rv_is_new, temp=t_opt, new_shift=b_opt),
        softmax_by_query(test_scores, rt_qcode, rt_is_new, temp=t_opt, new_shift=b_opt),
    )

def fit_ranker_member(name, kind):
    t0 = time.time()
    seeds = (SEED, SEED + 1) if kind == "lgb" else (SEED,)
    oof_list, test_list = [], []
    for sd in seeds:
        oof = np.zeros(len(RV), dtype=np.float64)
        for tr, va in GroupKFold(5).split(RV, y_rv, rv_groups):
            R_tr = RV.iloc[tr]
            if kind == "lgb":
                m = lgb.LGBMRanker(
                    objective="lambdarank", n_estimators=550, learning_rate=0.03,
                    num_leaves=31, min_child_samples=50, subsample=0.8, subsample_freq=1,
                    colsample_bytree=0.75, reg_alpha=1.0, reg_lambda=5.0,
                    lambdarank_truncation_level=8, verbose=-1, random_state=sd, n_jobs=N_WORKERS
                ).fit(R_tr[RANK_FEATS], y_rv[tr], group=R_tr.groupby("QKey", sort=False).size().to_numpy())
            else:
                m = xgb.XGBRanker(
                    objective="rank:ndcg", eval_metric="map@5", tree_method="hist",
                    n_estimators=450, learning_rate=0.04, max_depth=5, min_child_weight=10,
                    subsample=0.8, colsample_bytree=0.75, reg_alpha=1.0, reg_lambda=5.0,
                    random_state=sd, n_jobs=N_WORKERS
                ).fit(R_tr[RANK_FEATS], y_rv[tr], qid=pd.factorize(R_tr["QKey"])[0])
            oof[va] = m.predict(RV.iloc[va][RANK_FEATS])
        # Full fit for test
        if kind == "lgb":
            m_full = lgb.LGBMRanker(
                objective="lambdarank", n_estimators=550, learning_rate=0.03,
                num_leaves=31, min_child_samples=50, subsample=0.8, subsample_freq=1,
                colsample_bytree=0.75, reg_alpha=1.0, reg_lambda=5.0,
                lambdarank_truncation_level=8, verbose=-1, random_state=sd, n_jobs=N_WORKERS
            ).fit(RV[RANK_FEATS], y_rv, group=RV.groupby("QKey", sort=False).size().to_numpy())
        else:
            m_full = xgb.XGBRanker(
                objective="rank:ndcg", eval_metric="map@5", tree_method="hist",
                n_estimators=450, learning_rate=0.04, max_depth=5, min_child_weight=10,
                subsample=0.8, colsample_bytree=0.75, reg_alpha=1.0, reg_lambda=5.0,
                random_state=sd, n_jobs=N_WORKERS
            ).fit(RV[RANK_FEATS], y_rv, qid=pd.factorize(RV["QKey"])[0])
        oof_list.append(oof)
        test_list.append(m_full.predict(RT[RANK_FEATS]))

    oof_avg = np.mean(oof_list, axis=0)
    test_avg = np.mean(test_list, axis=0)
    # Per-query standardize before softmax calibration
    def _qstd(x, qc):
        s = pd.Series(x).groupby(qc)
        return (x - s.transform("mean").to_numpy()) / (s.transform("std").to_numpy() + 1e-6)
    p_oof, p_test = calibrate_ranker_probs(_qstd(oof_avg, rv_qcode), _qstd(test_avg, rt_qcode))
    print(f"  Trained {name:10s} in {time.time() - t0:.1f}s")
    return p_oof, p_test

print("Training diverse ensemble members out-of-fold...")
MEMBERS_OOF, MEMBERS_TEST = {}, {}

# 1. LightGBM 2-stage classifier (already fitted in Stage 2!)
MEMBERS_OOF["lgb_clf"] = clf_to_row_probs(V2, p_new_v2, RV)
MEMBERS_TEST["lgb_clf"] = clf_to_row_probs(T2, p_new_t2, RT)

# 2. CatBoost 2-stage classifier (symmetric oblivious trees)
MEMBERS_OOF["cb_clf"], MEMBERS_TEST["cb_clf"] = fit_two_stage_clf(
    "cb_clf",
    lambda: CatBoostClassifier(
        iterations=600, learning_rate=0.04, depth=6, l2_leaf_reg=5.0,
        verbose=0, random_seed=SEED, thread_count=N_WORKERS
    )
)

# 3. Regularized MLP 2-stage classifier (smooth non-tree diversity)
MEMBERS_OOF["mlp_clf"], MEMBERS_TEST["mlp_clf"] = fit_two_stage_clf(
    "mlp_clf",
    lambda: make_pipeline(
        SimpleImputer(strategy="median", add_indicator=True),
        StandardScaler(),
        MLPClassifier(hidden_layer_sizes=(64, 32), alpha=5e-2, max_iter=150,
                      early_stopping=True, random_state=SEED)
    )
)

# 4. LightGBM LambdaRank (2 seeds)
MEMBERS_OOF["lgb_rank"], MEMBERS_TEST["lgb_rank"] = fit_ranker_member("lgb_rank", "lgb")

# 5. XGBoost NDCG Ranker
MEMBERS_OOF["xgb_rank"], MEMBERS_TEST["xgb_rank"] = fit_ranker_member("xgb_rank", "xgb")

def eval_row_probs_map5(R_df, probs):
    code, keys = pd.factorize(R_df["QKey"])
    order = np.lexsort((-probs, code))
    g = code[order]
    pos = np.arange(len(g)) - np.searchsorted(g, g)
    hit = (R_df["Identity"].to_numpy() == R_df["Truth"].to_numpy())[order] & (pos < 5)
    per_q = np.zeros(len(keys))
    per_q[g[hit]] = 1.0 / (pos[hit] + 1)
    return float(per_q.mean())

print("\nIndividual Member OOF CV MAP@5:")
for name, p_oof in MEMBERS_OOF.items():
    print(f"  {name:10s}: {eval_row_probs_map5(RV, p_oof):.5f}")

# ---- Convex Simplex Blending via Out-of-Fold Log-Loss ----
m_names = list(MEMBERS_OOF.keys())
P_oof_mat = np.column_stack([MEMBERS_OOF[n] for n in m_names])
P_test_mat = np.column_stack([MEMBERS_TEST[n] for n in m_names])
pos_idx = (y_rv == 1)

def blend_logloss(raw_w):
    w = np.exp(raw_w - np.max(raw_w))
    w /= w.sum()
    p = np.clip(P_oof_mat[pos_idx] @ w, 1e-9, 1.0)
    return -float(np.mean(np.log(p)))

res_w = minimize(blend_logloss, x0=np.zeros(len(m_names)), method="BFGS")
W_OPT = np.exp(res_w.x - np.max(res_w.x))
W_OPT /= W_OPT.sum()

P_ens_oof = P_oof_mat @ W_OPT
P_ens_test = P_test_mat @ W_OPT

print("\nSimplex Log-Loss Optimal Weights:")
for n, w in zip(m_names, W_OPT):
    print(f"  {n:10s}: {w:.3f}")
print(f"Ensemble OOF CV MAP@5 (before prior shift): {eval_row_probs_map5(RV, P_ens_oof):.5f}")

### 1.7 Per-Species Prior-Shift Adjustment (Saerens EM) & Final Submission

Hyenas and leopards can have different `new_individual` proportions in the hidden test set. Using Saerens et al. (2002) EM separately per species gives the exact Bayes log-odds adjustment with zero overfitting.

In [ ]:
def saerens_em(p_test_new, val_prior, n_iter=200):
    p = float(val_prior)
    for _ in range(n_iter):
        w = (p / val_prior) / ((1.0 - p) / (1.0 - val_prior))
        p = float(np.mean((p_test_new * w) / (p_test_new * w + 1.0 - p_test_new)))
    return float(np.clip(p, 0.03, 0.70))

def apply_species_prior_shift(R_df, probs, shifts_by_sp):
    code = pd.factorize(R_df["QKey"])[0]
    is_new = (R_df["IsNewRow"].to_numpy() == 1)
    sp_arr = R_df["SpeciesC"].to_numpy()
    adj = probs.copy()
    for sp_idx, mult in shifts_by_sp.items():
        mask = is_new & (sp_arr == sp_idx)
        adj[mask] *= mult
    denom = pd.Series(adj).groupby(code).transform("sum").to_numpy()
    return adj / np.maximum(denom, 1e-12)

shifts_by_sp = {}
print("Per-species new_individual prior shift (Saerens EM):")
for sp_idx, sp_name in enumerate(SPECIES_LIST):
    v_mask = rv_is_new & (RV["SpeciesC"].to_numpy() == sp_idx)
    t_mask = rt_is_new & (RT["SpeciesC"].to_numpy() == sp_idx)
    val_prior = float(y_rv[v_mask].mean())
    est_test_prior = saerens_em(P_ens_test[t_mask], val_prior)
    # Dampened Bayes odds ratio (0.75 dampening guards against slight EM over-correction)
    odds_ratio = ((est_test_prior / (1.0 - est_test_prior)) / (val_prior / (1.0 - val_prior))) ** 0.75
    shifts_by_sp[sp_idx] = float(odds_ratio)
    print(f"  {sp_name:10s} | val_new={val_prior:.1%} -> est_test_new={est_test_prior:.1%} | odds_mult={odds_ratio:.3f}")

P_final_test = apply_species_prior_shift(RT, P_ens_test, shifts_by_sp)

# Extract top-5 predictions per test query
order = np.lexsort((-P_final_test, rt_qcode))
g = rt_qcode[order]
pos = np.arange(len(g)) - np.searchsorted(g, g)
keep = order[pos < 5]
top_df = RT.iloc[keep][["Query", "Identity"]]
final_preds = top_df.groupby("Query", sort=False)["Identity"].apply(list).to_dict()

sub = check_and_save_submission(final_preds, filename="model1_improved_ensemble.csv")

# Final sanity diagnostics
top1_is_new = np.mean([preds[0] == NEW for preds in final_preds.values()])
in_top5_new = np.mean([NEW in preds for preds in final_preds.values()])
print(f"\nTest Diagnostics: top-1 is new_individual = {top1_is_new:.1%} | new_individual in top-5 = {in_top5_new:.1%}")
print(f"Total end-to-end runtime: {(time.time() - T_START) / 60.0:.2f} minutes")
sub.head(10)

In [ ]:
%reset -f
# Model 1's CSV is on disk; clear its objects to free memory so the next model starts clean.
import gc
gc.collect()

## Model 2 — XQDA + KDE / two-hop retrieval and a 5-member ensemble (not yet submitted)

The pipeline of `Wildlife_ReID_Final.ipynb` (models only). Differences from Model 1:
* a second metric space (**XQDA/KISSME**, fitted on gallery labels only) next to LDA, and two extra distance-based scores (**kernel density** per animal, **two-hop** nearest neighbour);
* features that mean the same thing in validation and test: hubness measured against a **fixed-size gallery sample**, and test queries split into **groups the size of a validation fold** wherever queries are compared with each other;
* query-to-query evidence only through **confirmed links** (connected components);
* the `new_individual` share in test estimated by prior-shift EM, a per-species `new_individual` shift tuned on CV re-weighted to that share, and a **held-out** check of the blend.

Output: `model2_xqda_kde_ensemble.csv`. Caches: `Work/m2_*.pkl`.

In [ ]:
import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_v, "1")

import gc
import time
import pickle
import warnings
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import cv2
import numpy as np
import pandas as pd
import networkx as nx
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
from scipy.linalg import eigh
from scipy.optimize import minimize
from scipy.special import expit, logit

from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import GroupKFold
from sklearn.neural_network import MLPClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

warnings.filterwarnings("ignore")
cv2.setNumThreads(1)

T_START = time.time()
NEW = "new_individual"
SEED = 42
N_WORKERS = max(1, min(8, os.cpu_count() or 4))
BASE_DIR = os.environ.get("REID_BASE", None)

def resolve_paths(base_override=None):
    candidates = []
    if base_override:
        candidates.append(Path(base_override))
    candidates += [
        Path.cwd(),
        Path.cwd() / "reid_student_data_main",
        Path.cwd() / "projectwild3",
        Path.cwd() / "projectwild3" / "reid_student_data_main",
        Path("/kaggle/input"),
    ]
    ki = Path("/kaggle/input")
    if ki.exists():
        for p in ki.rglob("train.csv"):
            candidates.insert(0, p.parent)

    data_dir = None
    for c in candidates:
        if (c / "train.csv").exists() and (c / "miewid_features.npy").exists():
            data_dir = c
            break
        if (c / "reid_student_data_main" / "train.csv").exists():
            data_dir = c / "reid_student_data_main"
            break
    if data_dir is None:
        raise FileNotFoundError("Could not locate train.csv and miewid_features.npy. Set BASE_DIR.")

    def find_sift(name):
        for p in [data_dir / name, data_dir.parent / name, Path.cwd() / name]:
            if p.exists():
                return p
        for p in data_dir.parent.rglob(name):
            return p
        raise FileNotFoundError(f"Could not locate {name}")

    work_dir = Path("/kaggle/working/Work") if Path("/kaggle/working").exists() else (Path.cwd() / "Work")
    work_dir.mkdir(parents=True, exist_ok=True)
    out_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
    return data_dir, find_sift("sift_features_train.npz"), find_sift("sift_features_test.npz"), work_dir, out_dir

DATA_DIR, SIFT_TRAIN_PATH, SIFT_TEST_PATH, WORK_DIR, OUT_DIR = resolve_paths(BASE_DIR)
print(f"DATA_DIR: {DATA_DIR} | WORK_DIR: {WORK_DIR} | WORKERS: {N_WORKERS}")


# ---------------- configuration ----------------
KDE_BANDWIDTH = 0.10   # kernel width (cosine units) of the per-animal kernel density score
TWOHOP_K = 5           # neighbouring queries used by the two-hop score
PRUNE_TOP_K = 40       # keep this many identity features if that costs at most 0.001 CV

In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
for df in (train, test):
    df["viewpoint"] = df["viewpoint"].fillna("unknown").astype(str)

all_meta = pd.concat([train[["id", "species", "viewpoint"]],
                      test[["id", "species", "viewpoint"]]], ignore_index=True)
SP_MAP = dict(zip(all_meta["id"], all_meta["species"]))
VP_MAP = dict(zip(all_meta["id"], all_meta["viewpoint"]))
LAB_MAP = dict(zip(train["id"], train["identity"]))
SPECIES_LIST = sorted(train["species"].unique().tolist())

F_RAW = np.load(DATA_DIR / "miewid_features.npy").astype(np.float32)
MIEW_IDS = pd.read_csv(DATA_DIR / "miewid_ids.csv")["id"].tolist()
assert len(MIEW_IDS) == len(F_RAW)
ROW_MAP = {img_id: idx for idx, img_id in enumerate(MIEW_IDS)}
F_UNIT = F_RAW / (np.linalg.norm(F_RAW, axis=1, keepdims=True) + 1e-12)

SIDES = {
    "left": "L", "frontleft": "L", "backleft": "L",
    "right": "R", "frontright": "R", "backright": "R",
}
SIDE_CODE = {"L": 0, "R": 1, "O": 2, "U": 3}

def get_side(vp):
    return "U" if vp == "unknown" else SIDES.get(vp, "O")

def view_relation(vp_a, vp_b):
    sa, sb = get_side(vp_a), get_side(vp_b)
    if "U" in (sa, sb):
        return 3
    if sa == sb:
        return 0
    return 2 if {sa, sb} == {"L", "R"} else 1

def map5_score(truth_dict, preds_dict):
    tot = 0.0
    for q, t in truth_dict.items():
        for k, lab in enumerate(preds_dict.get(q, [])[:5]):
            if lab == t:
                tot += 1.0 / (k + 1)
                break
    return tot / max(len(truth_dict), 1)

def check_and_save_submission(preds_dict, filename="submission.csv"):
    sp_ident = dict(zip(train["identity"], train["species"]))
    sp_test = dict(zip(test["id"], test["species"]))
    rows = []
    for img_id in test["id"]:
        labs = []
        for l in preds_dict.get(img_id, []):
            if l not in labs and (l == NEW or sp_ident.get(l) == sp_test[img_id]):
                labs.append(l)
        if not labs:
            labs = [NEW]
        rows.append((img_id, " ".join(labs[:5])))
    sub = pd.DataFrame(rows, columns=["id", "predictions"])
    assert len(sub) == len(test) and set(sub["id"]) == set(test["id"])
    assert not sub["id"].duplicated().any()
    out_path = OUT_DIR / filename
    sub.to_csv(out_path, index=False)
    print(f"Saved valid submission -> {out_path} ({len(sub)} rows)")
    return sub

print(f"Train: {len(train)} imgs, {train['identity'].nunique()} IDs | Test: {len(test)} imgs")

### 2.1 Embeddings and validation design

* **Power-whitened PCA** (per species, fitted on train + test vectors without labels) removes the dominant directions of the MiewID vectors.
* **Encounters:** an animal's photos are linked when their whitened cosine is above 0.80 (connected components = graph clustering), grouping near-identical shots from one burst.
* **5 disjoint folds:** each animal is hidden completely in one fold (its photos become `new_individual` queries); for the other animals whole **encounters** are held out as queries, so a query never has a near-duplicate left in the gallery. The gallery keeps about **57–63 %** of train per fold.
* Because the test gallery holds 100 % of train while a fold gallery holds about 60 %, statistics that depend on gallery size (hubness) are computed against a **fixed-size gallery sample**, and test queries are split into **groups the size of a validation fold** wherever queries are compared with each other.

In [ ]:
def compute_power_whitened_pca(F, row_map, sp_map, n_comp=512, alpha=0.50, eps=1e-4):
    ids = sorted(row_map, key=row_map.get)
    spa = np.array([sp_map[i] for i in ids])
    E = np.zeros((len(F), n_comp), dtype=np.float32)
    for s in SPECIES_LIST:
        m = (spa == s)
        Z = F[m].astype(np.float64)
        Z = Z - Z.mean(axis=0, keepdims=True)
        k = min(n_comp, Z.shape[0] - 1, Z.shape[1])
        pca = PCA(n_components=k, svd_solver="randomized", random_state=SEED)
        U = pca.fit_transform(Z)
        U *= np.power(pca.explained_variance_ + eps, -0.5 * alpha)
        E[m, :k] = U.astype(np.float32)
    E /= (np.linalg.norm(E, axis=1, keepdims=True) + 1e-12)
    return E

def build_encounters(train_df, W, row_map, tau=0.80):
    enc = {}
    for ident, g in train_df.groupby("identity"):
        ids = g["id"].tolist()
        if len(ids) == 1:
            enc[ids[0]] = f"{ident}#0"
            continue
        X = W[[row_map[i] for i in ids]]
        _, comp = connected_components(csr_matrix((X @ X.T) > tau), directed=False)
        for img_id, c in zip(ids, comp):
            enc[img_id] = f"{ident}#{c}"
    return enc

def make_disjoint_folds(train_df, enc_map, n_folds=5, cap_unk=6, cap_known=8, seed=SEED):
    rng = np.random.default_rng(seed)
    lab = dict(zip(train_df["id"], train_df["identity"]))
    unk_fold = {}
    for _, d in train_df.groupby("species"):
        idents = rng.permutation(d["identity"].unique())
        for idx, ident in enumerate(idents):
            unk_fold[ident] = idx % n_folds

    enc_fold = {}
    for ident, g in train_df.groupby("identity"):
        uf = unk_fold[ident]
        encs = rng.permutation(sorted({enc_map[i] for i in g["id"]}))
        if len(encs) >= 2:
            avail = [f for f in range(n_folds) if f != uf]
            # Assign at most 1/5th of encounters per fold (keep >=1 encounter always in gallery)
            for idx, e in enumerate(encs):
                enc_fold[e] = avail[idx % len(avail)]

    folds = []
    for f in range(n_folds):
        queries, hidden = [], set()
        for ident, g in train_df.groupby("identity"):
            ids = g["id"].to_numpy()
            if unk_fold[ident] == f:
                hidden.update(ids)
                q_pick = rng.choice(ids, min(cap_unk, len(ids)), replace=False)
                queries.extend(q_pick.tolist())
            else:
                qi = [i for i in ids if enc_fold.get(enc_map[i], -1) == f]
                if 0 < len(qi) < len(ids):
                    hidden.update(qi)
                    q_pick = rng.choice(qi, min(cap_known, len(qi)), replace=False)
                    queries.extend(q_pick.tolist())
        qset = set(queries)
        gallery = [i for i in train_df["id"] if i not in qset and i not in hidden]
        gal_idents = {lab[i] for i in gallery}
        truth = {q: (lab[q] if lab[q] in gal_idents else NEW) for q in sorted(qset)}
        folds.append({"fold": f, "queryIds": sorted(qset), "galleryIds": gallery, "truth": truth})
    return folds

t0 = time.time()
W_PWR = compute_power_whitened_pca(F_RAW, ROW_MAP, SP_MAP, n_comp=512, alpha=0.50)
W_FULL = compute_power_whitened_pca(F_RAW, ROW_MAP, SP_MAP, n_comp=512, alpha=0.85)
ENC_MAP = build_encounters(train, W_PWR, ROW_MAP, tau=0.80)
FOLDS = make_disjoint_folds(train, ENC_MAP, n_folds=5, cap_unk=6, cap_known=8)

print(f"Built embeddings in {time.time() - t0:.1f}s | {len(set(ENC_MAP.values()))} encounters")
for f in FOLDS:
    t = pd.Series(f["truth"])
    print(f"  Fold {f['fold']}: {len(t):4d} queries | new={np.mean(t == NEW):.1%} | gallery={len(f['galleryIds']):4d} ({len(f['galleryIds'])/len(train):.0%})")


# Fixed-size gallery sample for hubness statistics: the same pool size in every fold and in test
POOL_SIZE = {s: min(sum(SP_MAP[i] == s for i in f["galleryIds"]) for f in FOLDS) for s in SPECIES_LIST}

# Test queries are split at random into groups the size of a validation fold, so every feature that
# compares queries with each other sees the same number of queries in validation and in test
fold_q = {s: np.mean([sum(SP_MAP[q] == s for q in f["queryIds"]) for f in FOLDS]) for s in SPECIES_LIST}
TEST_GROUP_OF = {}
rng_groups = np.random.default_rng(SEED)
for s in SPECIES_LIST:
    ids_s = rng_groups.permutation(test.loc[test["species"] == s, "id"].to_numpy())
    n_groups = max(1, int(round(len(ids_s) / fold_q[s])))
    for gi, chunk in enumerate(np.array_split(ids_s, n_groups)):
        for q in chunk:
            TEST_GROUP_OF[q] = gi
    print(f"  {s}: hubness pool = {POOL_SIZE[s]} gallery photos | test split into {n_groups} "
          f"query groups of ~{len(ids_s) / n_groups:.0f} (validation fold: ~{fold_q[s]:.0f})")

### 2.2 Candidate retrieval

For every query, the 45 most likely animals of the same species are kept (38 from the combined score + 7 from raw MiewID cosine). Scores per candidate animal:
* **CSLS** (hubness-corrected cosine) in the LDA space and in the **XQDA/KISSME** space. XQDA solves $\Sigma_E v = \lambda \Sigma_I v$ with $\Sigma_I$ from cross-encounter same-animal pairs and $\Sigma_E$ from hard impostor pairs (fitted on gallery labels only).
* **k-reciprocal Jaccard** similarity (computed within query groups).
* **Kernel density score:** $\mathrm{KDE}(q, a) = \frac{1}{n_a}\sum_{g \in a} \exp\big((\cos(q,g) - 1)/h\big)$ — how well the query fits the animal's photos as a whole.
* **Two-hop nearest neighbour:** $\max_{n \in \mathrm{kNN}(q)} \cos(q,n)\cdot \max_{g \in a}\cos(n,g)$ over neighbouring *queries* $n$ (within the same query group) — reaches an animal through an easier photo of the same animal.
* Same-flank, opposite-flank and other-view similarity, number of encounters, gallery size.

In [ ]:
TOP_ID = 45
RAW_SLOTS = 7
TOP_IMG_BASE = 3
TOP_IMG_DEEP_K = 6
TOP_IMG_DEEP_CAP = 16

def fit_xqda_projection(W, row_map, sp_map, lab_map, enc_map, gallery_ids, n_comp=256, k_hard=15, reg_i=0.05, reg_e=0.02):
    """
    Cross-Encounter Hard-Negative XQDA / KISSME linear metric projection.
    Fitted on gallery images only, per species: a linear metric learned from same-animal vs impostor pairs.
    """
    ids = sorted(row_map, key=row_map.get)
    spa = np.array([sp_map[i] for i in ids])
    d = W.shape[1]
    X_out = np.zeros((len(W), n_comp), dtype=np.float32)
    rng = np.random.default_rng(SEED)

    for s in SPECIES_LIST:
        g_ids = [i for i in gallery_ids if sp_map[i] == s]
        X_g = W[[row_map[i] for i in g_ids]].astype(np.float64)
        y_g = np.array([lab_map[i] for i in g_ids])
        e_g = np.array([enc_map[i] for i in g_ids])
        ng = len(g_ids)

        # ---- 1. Intra-class scatter Sigma_I from CROSS-ENCOUNTER same-ID pairs ----
        pos_i, pos_j, pos_w = [], [], []
        by_id = {}
        for idx, ident in enumerate(y_g):
            by_id.setdefault(ident, []).append(idx)

        for ident, idxs in by_id.items():
            if len(idxs) < 2:
                continue
            cross_pairs = []
            same_enc_pairs = []
            # Cap sampled indices per identity to avoid O(m^2) explosion
            sub_idxs = idxs if len(idxs) <= 30 else rng.choice(idxs, 30, replace=False).tolist()
            for a_pos in range(len(sub_idxs)):
                ia = sub_idxs[a_pos]
                for b_pos in range(a_pos + 1, len(sub_idxs)):
                    ib = sub_idxs[b_pos]
                    if e_g[ia] != e_g[ib]:
                        cross_pairs.append((ia, ib))
                    else:
                        same_enc_pairs.append((ia, ib))
            if cross_pairs:
                if len(cross_pairs) > 60:
                    sel = rng.choice(len(cross_pairs), 60, replace=False)
                    cross_pairs = [cross_pairs[k] for k in sel]
                w_pair = 1.0 / len(cross_pairs)
                for ia, ib in cross_pairs:
                    pos_i.append(ia); pos_j.append(ib); pos_w.append(w_pair)
            elif same_enc_pairs:
                # Fallback with lower weight for single-encounter identities
                if len(same_enc_pairs) > 20:
                    sel = rng.choice(len(same_enc_pairs), 20, replace=False)
                    same_enc_pairs = [same_enc_pairs[k] for k in sel]
                w_pair = 0.25 / len(same_enc_pairs)
                for ia, ib in same_enc_pairs:
                    pos_i.append(ia); pos_j.append(ib); pos_w.append(w_pair)

        pos_w = np.array(pos_w, dtype=np.float64)
        D_pos = (X_g[pos_i] - X_g[pos_j]) * np.sqrt(pos_w)[:, None]
        Sigma_I = (D_pos.T @ D_pos) / max(pos_w.sum(), 1e-9)

        # ---- 2. Extra-class scatter Sigma_E from TOP-K HARD NEGATIVE impostor pairs ----
        S_gg = X_g @ X_g.T
        same_mask = (y_g[:, None] == y_g[None, :])
        S_gg[same_mask] = -2.0
        kk = min(k_hard, ng - 1)
        hard_idx = np.argpartition(-S_gg, kk, axis=1)[:, :kk]
        neg_i = np.repeat(np.arange(ng), kk)
        neg_j = hard_idx.ravel()
        # Weight harder negatives slightly more
        neg_sims = np.clip(S_gg[neg_i, neg_j], 0.0, 1.0)
        neg_w = 0.5 + neg_sims
        D_neg = (X_g[neg_i] - X_g[neg_j]) * np.sqrt(neg_w)[:, None]
        Sigma_E = (D_neg.T @ D_neg) / max(neg_w.sum(), 1e-9)

        # Regularize both covariances proportionally to trace
        Sigma_I += reg_i * (np.trace(Sigma_I) / d) * np.eye(d)
        Sigma_E += reg_e * (np.trace(Sigma_E) / d) * np.eye(d)

        # Generalized eigendecomposition: Sigma_E v = lambda Sigma_I v
        eigvals, eigvecs = eigh(Sigma_E, Sigma_I)
        # Keep top eigenvectors with largest separation ratio lambda > 1
        ord_idx = np.argsort(-eigvals)[:n_comp]
        vals = np.maximum(eigvals[ord_idx] - 1.0, 1e-4)
        V = eigvecs[:, ord_idx] * np.sqrt(vals)[None, :]  # (d, n_comp)

        m = (spa == s)
        proj = (W[m].astype(np.float64) @ V).astype(np.float32)
        proj /= (np.linalg.norm(proj, axis=1, keepdims=True) + 1e-12)
        X_out[m] = proj

    return X_out

def fit_gallery_lda_blend(W, row_map, sp_map, lab_map, gallery_ids, n_comp=160, mix=0.50):
    ids = sorted(row_map, key=row_map.get)
    spa = np.array([sp_map[i] for i in ids])
    L = np.zeros((len(W), n_comp), dtype=np.float32)
    for s in SPECIES_LIST:
        g = [i for i in gallery_ids if sp_map[i] == s]
        y = np.array([lab_map[i] for i in g])
        vc = pd.Series(y).value_counts()
        valid_ids = set(vc[vc >= 2].index)
        g_fit = [i for i in g if lab_map[i] in valid_ids]
        if len(valid_ids) < 3:
            g_fit = g
        y_fit = np.array([lab_map[i] for i in g_fit])
        k = min(n_comp, len(np.unique(y_fit)) - 1, W.shape[1])
        lda = LinearDiscriminantAnalysis(n_components=k, solver="eigen", shrinkage="auto")
        lda.fit(W[[row_map[i] for i in g_fit]], y_fit)
        L[spa == s, :k] = lda.transform(W[spa == s]).astype(np.float32)
    L /= (np.linalg.norm(L, axis=1, keepdims=True) + 1e-12)
    E = np.hstack([np.sqrt(1.0 - mix) * W, np.sqrt(mix) * L]).astype(np.float32)
    E /= (np.linalg.norm(E, axis=1, keepdims=True) + 1e-12)
    return E

def compute_k_reciprocal_jaccard(E_q, E_g, k1=15, k2=5):
    X = np.vstack([E_q, E_g]).astype(np.float32)
    N, nq = len(X), len(E_q)
    S = X @ X.T
    k1_eff = min(k1, N - 1)
    top_idx = np.argpartition(-S, k1_eff, axis=1)[:, :k1_eff + 1]
    rows = np.repeat(np.arange(N), k1_eff + 1)
    cols = top_idx.ravel()
    vals = np.exp(np.clip(S[rows, cols] * 5.0, -20, 20)).astype(np.float32)
    A = csr_matrix((vals, (rows, cols)), shape=(N, N))
    R = A.minimum(A.T).tocsr()
    k2_eff = min(k2, N - 1)
    top_k2 = np.argpartition(-S, k2_eff, axis=1)[:, :k2_eff + 1]
    r2 = np.repeat(np.arange(N), k2_eff + 1)
    c2 = top_k2.ravel()
    Qexp = csr_matrix((np.full(len(r2), 1.0 / (k2_eff + 1), dtype=np.float32), (r2, c2)), shape=(N, N))
    V = (Qexp @ R).tocsr()
    row_norms = np.sqrt(np.asarray(V.multiply(V).sum(axis=1)).ravel()) + 1e-12
    V = V.multiply(1.0 / row_norms[:, None]).tocsr()
    return (V[:nq] @ V[nq:].T).toarray().astype(np.float32)

def kde_identity_scores(S_c, starts, counts, h=KDE_BANDWIDTH):
    """Kernel density estimate of each query under each animal's gallery photos
    (Gaussian kernel on cosine distance, averaged over the animal's photos)."""
    K = np.exp((S_c - 1.0) / h)
    return np.add.reduceat(K, starts, axis=1) / counts[None, :]


def two_hop_scores(C_q, M_c, k=TWOHOP_K):
    """Best path query -> neighbouring query n -> animal: max_n cos(q, n) * M_c[n, animal]
    (k-NN / distance-based retrieval). C_q holds the queries of ONE query group only."""
    nq = len(C_q)
    if nq < 2:
        return np.zeros_like(M_c)
    S_qq = C_q @ C_q.T
    np.fill_diagonal(S_qq, -1.0)
    kk = min(k, nq - 1)
    idx = np.argpartition(-S_qq, kk - 1, axis=1)[:, :kk]
    w = np.clip(np.take_along_axis(S_qq, idx, axis=1), 0.0, 1.0)
    return (w[:, :, None] * M_c[idx]).max(axis=1)


def hub_stats(S_qg, S_gg, seg, in_pool, k):
    """CSLS hubness terms measured against a FIXED-SIZE gallery sample (same pool size in CV and test)."""
    pool = np.flatnonzero(in_pool)
    kk = min(k, len(pool) - 1)
    S_gp = S_gg[:, pool].copy()
    S_gp[seg[:, None] == seg[pool][None, :]] = -1.0
    g_hub = np.partition(S_gp, -kk, axis=1)[:, -kk:].mean(axis=1)
    q_hub = np.partition(S_qg[:, pool], -kk, axis=1)[:, -kk:].mean(axis=1)
    return q_hub, g_hub


def retrieve_candidates(E_lda, E_xqda, W_pwr, W_full, F_unit, row_map, sp_map, vp_map, enc_map,
                        query_ids, gallery_df, pool_size, group_of=None):
    out_rows = []
    E_comb = np.hstack([np.sqrt(0.55) * E_lda, np.sqrt(0.45) * E_xqda]).astype(np.float32)
    E_comb /= (np.linalg.norm(E_comb, axis=1, keepdims=True) + 1e-12)

    for s, G in gallery_df.groupby("species"):
        qids = [q for q in query_ids if sp_map[q] == s]
        if not qids:
            continue
        G = G.sort_values("identity").reset_index(drop=True)
        gid = G["id"].to_numpy()
        glab = G["identity"].to_numpy()
        gside = np.array([get_side(vp_map[g]) for g in gid])
        genc = np.array([enc_map[g] for g in gid])
        starts = np.r_[0, np.flatnonzero(glab[1:] != glab[:-1]) + 1]
        counts = np.diff(np.r_[starts, len(glab)])
        seg = np.repeat(np.arange(len(starts)), counts)
        labs = glab[starts]
        n_id = len(starts)

        # fixed-size gallery sample for hubness; query groups (one in CV, fold-sized groups in test)
        rng = np.random.default_rng(SEED)
        in_pool = np.zeros(len(gid), dtype=bool)
        in_pool[rng.choice(len(gid), min(pool_size[s], len(gid)), replace=False)] = True
        if group_of is None:
            groups = [np.arange(len(qids))]
        else:
            gq = np.array([group_of[q] for q in qids])
            groups = [np.flatnonzero(gq == v) for v in np.unique(gq)]

        q_idx = [row_map[q] for q in qids]
        g_idx = [row_map[g] for g in gid]
        Eq, Eg = E_lda[q_idx], E_lda[g_idx]
        Xq, Xg = E_xqda[q_idx], E_xqda[g_idx]
        Cq, Cg = E_comb[q_idx], E_comb[g_idx]
        Wq, Wg = W_pwr[q_idx], W_pwr[g_idx]
        WFq, WFg = W_full[q_idx], W_full[g_idx]
        Rq, Rg = F_unit[q_idx], F_unit[g_idx]

        # identity centroids in the combined space
        centroids = np.zeros((n_id, Cg.shape[1]), dtype=np.float32)
        for j in range(n_id):
            c_vec = Cg[starts[j]:starts[j] + counts[j]].mean(axis=0)
            centroids[j] = c_vec / (np.linalg.norm(c_vec) + 1e-12)
        S_cent = Cq @ centroids.T

        # ---- space 1: LDA + whitened, CSLS ----
        k_hub = 10
        S = Eq @ Eg.T
        q_hub, g_hub = hub_stats(S, Eg @ Eg.T, seg, in_pool, k_hub)
        S_csls = 2.0 * S - q_hub[:, None] - g_hub[None, :]

        # ---- space 2: cross-encounter XQDA/KISSME, CSLS ----
        SX = Xq @ Xg.T
        qx_hub, gx_hub = hub_stats(SX, Xg @ Xg.T, seg, in_pool, k_hub)
        SX_csls = 2.0 * SX - qx_hub[:, None] - gx_hub[None, :]

        # ---- space 3 (combined): kernel density, k-reciprocal Jaccard and two-hop (per query group) ----
        S_c = Cq @ Cg.T
        M_c = np.maximum.reduceat(S_c, starts, axis=1)
        KDE = kde_identity_scores(S_c, starts, counts)
        KDE_share = KDE / (KDE.sum(axis=1, keepdims=True) + 1e-12)
        J_mat = np.zeros((len(qids), len(gid)), dtype=np.float32)
        TH = np.zeros((len(qids), n_id), dtype=np.float32)
        for gq_idx in groups:
            J_mat[gq_idx] = compute_k_reciprocal_jaccard(Cq[gq_idx], Cg, k1=15, k2=5)
            TH[gq_idx] = two_hop_scores(Cq[gq_idx], M_c[gq_idx])

        q_mean, q_std = S.mean(axis=1), S.std(axis=1) + 1e-6
        qx_mean, qx_std = SX.mean(axis=1), SX.std(axis=1) + 1e-6

        M_main = np.maximum.reduceat(S, starts, axis=1)
        M_csls = np.maximum.reduceat(S_csls, starts, axis=1)
        M_xcsls = np.maximum.reduceat(SX_csls, starts, axis=1)
        M_jac = np.maximum.reduceat(J_mat, starts, axis=1)
        M_raw = np.maximum.reduceat(Rq @ Rg.T, starts, axis=1)
        M_wp = np.maximum.reduceat(Wq @ Wg.T, starts, axis=1)
        M_wf = np.maximum.reduceat(WFq @ WFg.T, starts, axis=1)

        R_csls = np.argsort(np.argsort(-M_csls, axis=1), axis=1)
        R_xcsls = np.argsort(np.argsort(-M_xcsls, axis=1), axis=1)
        R_jac = np.argsort(np.argsort(-M_jac, axis=1), axis=1)
        R_kde = np.argsort(np.argsort(-KDE, axis=1), axis=1)
        R_th = np.argsort(np.argsort(-TH, axis=1), axis=1)
        R_raw = np.argsort(np.argsort(-M_raw, axis=1), axis=1)
        R_wp = np.argsort(np.argsort(-M_wp, axis=1), axis=1)

        # candidate selection: pure metric scores (KDE scaled to [0, 1] per query)
        M_sel = (0.55 * M_csls + 0.45 * M_xcsls + 0.30 * M_jac
                 + 0.25 * KDE / (KDE.max(axis=1, keepdims=True) + 1e-12))

        for r, q in enumerate(qids):
            main_ord = np.argsort(-M_sel[r])
            order = main_ord[:TOP_ID - RAW_SLOTS]
            chosen = set(order.tolist())
            raw_extra = [j for j in np.argsort(-M_raw[r]) if j not in chosen][:RAW_SLOTS]
            order = np.r_[order, raw_extra].astype(int)

            S_comb_r = 0.55 * S_csls[r] + 0.45 * SX_csls[r]
            top15_g = np.argsort(-S_comb_r)[:15]
            knn15 = np.bincount(seg[top15_g], minlength=n_id)
            qs = get_side(vp_map[q])

            def top12(M):
                t1 = float(M[r].max())
                t2 = float(np.partition(M[r], -2)[-2]) if n_id > 1 else t1
                return t1, t1 - t2

            top1_cos, gap_cos = top12(M_main)
            top1_csls, gap_csls = top12(M_csls)
            top1_xcsls, gap_xcsls = top12(M_xcsls)
            top1_jac, gap_jac = top12(M_jac)
            top1_kde, gap_kde = top12(KDE)
            top1_th, gap_th = top12(TH)

            # same-flank query hubness, measured within the fixed-size pool
            sf_mask = (gside == qs) & in_pool if qs in "LR" else None
            if sf_mask is not None and sf_mask.sum() >= 10:
                q_sf_hub = float(np.partition(S[r, sf_mask], -10)[-10:].mean())
            else:
                q_sf_hub = float(q_hub[r])

            for rk, j in enumerate(order):
                a, n = starts[j], counts[j]
                v_cos = S[r, a:a + n]
                v_csls = S_csls[r, a:a + n]
                v_xcos = SX[r, a:a + n]
                v_xcsls = SX_csls[r, a:a + n]
                v_jac = J_mat[r, a:a + n]
                v_hub = g_hub[a:a + n]
                v_enc = genc[a:a + n]
                o = np.argsort(-(0.55 * v_csls + 0.45 * v_xcsls))

                enc_best = {}
                for idx_e, enc_id in enumerate(v_enc):
                    val_e = float(0.5 * v_cos[idx_e] + 0.5 * v_xcos[idx_e])
                    if enc_id not in enc_best or val_e > enc_best[enc_id]:
                        enc_best[enc_id] = val_e
                enc_vals = sorted(enc_best.values(), reverse=True)
                n_enc_tot = len(enc_vals)
                cos_2nd_enc = enc_vals[1] if n_enc_tot >= 2 else np.nan
                n_enc_strong = sum(1 for x in enc_vals if x >= 0.42)

                cap_img = TOP_IMG_DEEP_CAP if rk < TOP_IMG_DEEP_K else TOP_IMG_BASE
                best_imgs = list(gid[a + o[:cap_img]])

                gs = gside[a:a + n]
                same_idx = np.flatnonzero(gs == qs) if qs in "LR" else np.array([], dtype=int)
                opp_idx = np.flatnonzero(gs == {"L": "R", "R": "L"}.get(qs, "-"))
                oth_idx = np.flatnonzero(gs == "O")
                if len(same_idx):  # always SIFT-check the 3 best same-flank photos
                    sf_ord = same_idx[np.argsort(-(0.55 * v_csls[same_idx] + 0.45 * v_xcsls[same_idx]))[:3]]
                    for s_i in sf_ord:
                        if gid[a + s_i] not in best_imgs:
                            best_imgs.append(gid[a + s_i])

                same_max = float(v_cos[same_idx].max()) if len(same_idx) else np.nan
                same_xmax = float(v_xcos[same_idx].max()) if len(same_idx) else np.nan
                same_csls = float((2.0 * v_cos[same_idx] - q_sf_hub - v_hub[same_idx]).max()) if len(same_idx) else np.nan
                opp_max = float(v_cos[opp_idx].max()) if len(opp_idx) else np.nan
                oth_max = float(v_cos[oth_idx].max()) if len(oth_idx) else np.nan
                n_sides = len(set(gs) - {"U"})
                if qs in "LR":
                    n_comp = int((gs == qs).sum())
                elif qs == "O":
                    n_comp = int(sum(vp_map[g] == vp_map[q] for g in gid[a:a + n]))
                else:
                    n_comp = -1

                out_rows.append((
                    q, labs[j],
                    float(v_cos[o[0]]), float(v_cos[o[1]]) if n > 1 else np.nan, float(v_cos[o[:3]].mean()),
                    float(v_csls[o[0]]), float(v_csls[o[1]]) if n > 1 else np.nan, float(v_csls[o[:3]].mean()),
                    float(v_xcos[o[0]]), float(v_xcos[o[1]]) if n > 1 else np.nan, float(v_xcos[o[:3]].mean()),
                    float(v_xcsls[o[0]]), float(v_xcsls[o[1]]) if n > 1 else np.nan,
                    float(v_jac[o[0]]), float(KDE[r, j]), float(KDE_share[r, j]), float(TH[r, j]), float(S_cent[r, j]),
                    float((v_cos[o[0]] - q_mean[r]) / q_std[r]),
                    float((v_xcos[o[0]] - qx_mean[r]) / qx_std[r]),
                    float(v_hub[o[0]]), float(q_hub[r]), float(qx_hub[r]),
                    float(knn15[j] / 15.0),
                    cos_2nd_enc, n_enc_tot, n_enc_strong,
                    same_max, same_xmax, same_csls, opp_max, oth_max, n_sides,
                    rk, int(R_csls[r, j]), int(R_xcsls[r, j]), int(R_jac[r, j]), int(R_kde[r, j]), int(R_th[r, j]),
                    int(R_raw[r, j]), int(R_wp[r, j]),
                    top1_cos, gap_cos, top1_csls, gap_csls, top1_xcsls, gap_xcsls,
                    top1_jac, gap_jac, top1_kde, gap_kde, top1_th, gap_th,
                    int(n), n_comp,
                    float(M_raw[r, j]), float(M_wp[r, j]), float(M_wf[r, j]),
                    best_imgs,
                ))

    cols = [
        "Query", "Identity",
        "CosMax", "Cos2nd", "CosMean3",
        "CSLSMax", "CSLS2nd", "CSLSMean3",
        "XCosMax", "XCos2nd", "XCosMean3",
        "XCSLSMax", "XCSLS2nd",
        "JacMax", "KDEScore", "KDEShare", "TwoHopMax", "CentroidCos",
        "CosZ", "XCosZ", "GalHub", "QHub", "QXHub",
        "Knn15Frac",
        "Cos2ndEnc", "NEncTotal", "NEncStrong",
        "SameSideMax", "SameSideXMax", "SameSideCSLS", "OppSideMax", "OtherViewMax", "NSides",
        "IdRank", "CSLSRank", "XCSLSRank", "JacRank", "KDERank", "TwoHopRank", "RawRank", "WPRank",
        "QTop1Cos", "QGap12Cos", "QTop1CSLS", "QGap12CSLS", "QTop1XCSLS", "QGap12XCSLS",
        "QTop1Jac", "QGap12Jac", "QTop1KDE", "QGap12KDE", "QTop1TwoHop", "QGap12TwoHop",
        "NGal", "NCompGal",
        "RawCosMax", "WPCosMax", "WFCosMax",
        "BestImgs",
    ]
    return pd.DataFrame(out_rows, columns=cols)


t0 = time.time()
tr_sub = train[["id", "identity", "species"]]
FOLD_EMBS = {}
CAND_FOLDS = []

for f in FOLDS:
    E_lda = fit_gallery_lda_blend(W_PWR, ROW_MAP, SP_MAP, LAB_MAP, f["galleryIds"])
    E_xqda = fit_xqda_projection(W_PWR, ROW_MAP, SP_MAP, LAB_MAP, ENC_MAP, f["galleryIds"])
    E_comb = np.hstack([np.sqrt(0.55) * E_lda, np.sqrt(0.45) * E_xqda]).astype(np.float32)
    E_comb /= (np.linalg.norm(E_comb, axis=1, keepdims=True) + 1e-12)
    FOLD_EMBS[f["fold"]] = E_comb
    gal_df = tr_sub[tr_sub["id"].isin(set(f["galleryIds"]))]
    cf = retrieve_candidates(E_lda, E_xqda, W_PWR, W_FULL, F_UNIT, ROW_MAP, SP_MAP, VP_MAP, ENC_MAP,
                             f["queryIds"], gal_df, POOL_SIZE)
    cf["Fold"] = f["fold"]
    cf["QKey"] = f"{f['fold']}:" + cf["Query"]
    cf["Truth"] = cf["Query"].map(f["truth"])
    CAND_FOLDS.append(cf)
    print(f"  Fold {f['fold']} done ({time.time() - t0:.1f}s)")

E_LDA_T = fit_gallery_lda_blend(W_PWR, ROW_MAP, SP_MAP, LAB_MAP, train["id"].tolist())
E_XQDA_T = fit_xqda_projection(W_PWR, ROW_MAP, SP_MAP, LAB_MAP, ENC_MAP, train["id"].tolist())
E_TEST = np.hstack([np.sqrt(0.55) * E_LDA_T, np.sqrt(0.45) * E_XQDA_T]).astype(np.float32)
E_TEST /= (np.linalg.norm(E_TEST, axis=1, keepdims=True) + 1e-12)

CAND_TEST = retrieve_candidates(E_LDA_T, E_XQDA_T, W_PWR, W_FULL, F_UNIT, ROW_MAP, SP_MAP, VP_MAP, ENC_MAP,
                                test["id"].tolist(), tr_sub, POOL_SIZE, group_of=TEST_GROUP_OF)
CAND_TEST["QKey"] = CAND_TEST["Query"]

V_ALL = pd.concat(CAND_FOLDS, ignore_index=True)

recalls = []
for f, cf in zip(FOLDS, CAND_FOLDS):
    c_map = cf.groupby("Query")["Identity"].apply(set).to_dict()
    known_q = [q for q, t in f["truth"].items() if t != NEW]
    recalls.append(np.mean([f["truth"][q] in c_map.get(q, set()) for q in known_q]))
print(f"Retrieval done in {time.time() - t0:.1f}s | Mean Recall@{TOP_ID} (known animals): {np.mean(recalls):.4f}")

### 2.3 SIFT verification

RootSIFT descriptors, Lowe ratio test (0.80), one-to-one matching, then RANSAC (affine, falling back to similarity) on the provided keypoints. Checked pairs: up to 16 photos of the top-6 candidates, 3 photos of every other candidate in the top 35, the 3 best same-flank photos of every candidate, and confirmed query-to-query neighbours (built within query groups for test). Results are cached in `Work/m2_sift_rich_cache.pkl`.

In [ ]:
SIFT_RATIO = 0.80
RANSAC_PX = 14.0
Q_NBR_K = 12

def get_query_neighbors(E, row_map, qids, sp_map, k=Q_NBR_K, group_of=None):
    """k nearest other queries of the same species (and the same query group, if given)"""
    out = {}
    for s in SPECIES_LIST:
        qs_all = [q for q in qids if sp_map[q] == s]
        group_vals = sorted({group_of[q] for q in qs_all}) if group_of else [None]
        for gv in group_vals:
            qs = [q for q in qs_all if group_of is None or group_of[q] == gv]
            if len(qs) <= 1:
                continue
            X = E[[row_map[q] for q in qs]]
            S = X @ X.T
            np.fill_diagonal(S, -9.0)
            kk = min(k, len(qs) - 1)
            idx = np.argpartition(-S, kk, axis=1)[:, :kk]
            for a, q in enumerate(qs):
                nbrs = idx[a]
                nbrs = nbrs[np.argsort(-S[a, nbrs])]
                out[q] = [(qs[b], float(S[a, b])) for b in nbrs]
    return out

FOLD_QNBRS = {f["fold"]: get_query_neighbors(FOLD_EMBS[f["fold"]], ROW_MAP, f["queryIds"], SP_MAP) for f in FOLDS}
TEST_QNBRS = get_query_neighbors(E_TEST, ROW_MAP, test["id"].tolist(), SP_MAP, group_of=TEST_GROUP_OF)

req_pairs = set()
for df_c in [V_ALL, CAND_TEST]:
    sub_c = df_c[df_c["IdRank"] < 35]
    for q, imgs in zip(sub_c["Query"], sub_c["BestImgs"]):
        for g in imgs:
            req_pairs.add((q, g))

for nb_dict in list(FOLD_QNBRS.values()) + [TEST_QNBRS]:
    for q, lst in nb_dict.items():
        for q2, sim in lst[:10]:
            if sim > 0.30:
                req_pairs.add((q, q2) if q < q2 else (q2, q))

SIFT_CACHE_PATH = WORK_DIR / "m2_sift_rich_cache.pkl"
NKP_CACHE_PATH = WORK_DIR / "m2_nkp_cache.pkl"

if SIFT_CACHE_PATH.exists() and NKP_CACHE_PATH.exists():
    with open(SIFT_CACHE_PATH, "rb") as fh:
        SIFT_CACHE = pickle.load(fh)
    with open(NKP_CACHE_PATH, "rb") as fh:
        NKP_MAP = pickle.load(fh)
else:
    SIFT_CACHE, NKP_MAP = {}, {}

todo_pairs = [p for p in req_pairs if p not in SIFT_CACHE]
print(f"SIFT requested: {len(req_pairs):,} | Cached: {len(req_pairs) - len(todo_pairs):,} | Missing: {len(todo_pairs):,}")

def load_sift_store_fast(img_ids):
    train_set = set(train["id"])
    empty = (np.zeros((0, 2), np.float32), np.zeros((0, 128), np.float16))

    if SIFT_TRAIN_PATH.is_dir() and SIFT_TEST_PATH.is_dir():
        def _load_one(img_id):
            src = SIFT_TRAIN_PATH if img_id in train_set else SIFT_TEST_PATH
            fx, fd = src / f"{img_id}_xy.npy", src / f"{img_id}_desc.npy"
            if not (fx.exists() and fd.exists()):
                return img_id, empty
            xy, d = np.load(fx), np.load(fd)
            if len(d) == 0:
                return img_id, empty
            d = d.astype(np.float32).reshape(-1, 128)
            d /= (d.sum(axis=1, keepdims=True) + 1e-7)
            return img_id, (xy.astype(np.float32).reshape(-1, 2), np.sqrt(d).astype(np.float16))
        with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
            return dict(ex.map(_load_one, img_ids))

    npz_handles, store = {}, {}
    for img_id in img_ids:
        src = SIFT_TRAIN_PATH if img_id in train_set else SIFT_TEST_PATH
        if src not in npz_handles:
            npz_handles[src] = np.load(src)
        z = npz_handles[src]
        kx, kd = f"{img_id}_xy", f"{img_id}_desc"
        if kx in z.files and kd in z.files:
            xy, d = z[kx], z[kd]
            if len(d) > 0:
                d = d.astype(np.float32).reshape(-1, 128)
                d /= (d.sum(axis=1, keepdims=True) + 1e-7)
                store[img_id] = (xy.astype(np.float32).reshape(-1, 2), np.sqrt(d).astype(np.float16))
                continue
        store[img_id] = empty
    for h in npz_handles.values():
        h.close()
    return store

def match_pair_rich(qxy, qd32, gxy, gd16):
    if len(qd32) < 2 or len(gd16) < 2:
        return (0, 0, 0.0, 0.0, 0.0)
    gd32 = gd16.astype(np.float32)
    S = qd32 @ gd32.T
    r = np.arange(len(S))
    j1 = S.argmax(axis=1)
    s1 = S[r, j1].copy()
    S[r, j1] = -np.inf
    s2 = S.max(axis=1)
    d1 = np.sqrt(np.maximum(2.0 - 2.0 * s1, 0.0))
    d2 = np.sqrt(np.maximum(2.0 - 2.0 * s2, 0.0))
    qi = np.flatnonzero(d1 < SIFT_RATIO * d2)
    if len(qi) == 0:
        return (0, 0, 0.0, 0.0, 0.0)
    gj = j1[qi]
    o = np.argsort(d1[qi])
    qi, gj = qi[o], gj[o]
    _, u = np.unique(gj, return_index=True)
    qi, gj = qi[u], gj[u]
    n_good = int(len(qi))
    if n_good < 4:
        return (n_good, 0, 0.0, 0.0, 0.0)

    pts_q, pts_g = qxy[qi], gxy[gj]
    M, inl = cv2.estimateAffine2D(
        pts_q, pts_g, method=cv2.RANSAC,
        ransacReprojThreshold=RANSAC_PX, maxIters=2000, confidence=0.995
    )
    use_partial = True
    if M is not None and inl is not None:
        A = M[:, :2]
        det = float(np.linalg.det(A))
        sv = np.linalg.svd(A, compute_uv=False)
        if det > 0.05 and sv[1] > 0.2 and sv[0] < 4.5 and (sv[0] / (sv[1] + 1e-6)) < 3.5:
            use_partial = False
    if use_partial:
        _, inl = cv2.estimateAffinePartial2D(
            pts_q, pts_g, method=cv2.RANSAC,
            ransacReprojThreshold=RANSAC_PX, maxIters=2000, confidence=0.995
        )
    if inl is None:
        return (n_good, 0, 0.0, 0.0, 0.0)
    mask = inl.ravel().astype(bool)
    n_inl = int(mask.sum())
    if n_inl < 3:
        return (n_good, n_inl, float(n_inl / n_good), 0.0, 0.0)
    mean_sim = float(s1[qi[mask]].mean())
    q_span = float(np.hypot(qxy[:, 0].std(), qxy[:, 1].std()) + 1e-5)
    inl_pts = pts_q[mask]
    inl_spread = float(np.hypot(inl_pts[:, 0].std(), inl_pts[:, 1].std()) / q_span)
    return (n_good, n_inl, float(n_inl / n_good), mean_sim, inl_spread)

if todo_pairs:
    t0 = time.time()
    need_ids = sorted({i for p in todo_pairs for i in p})
    sift_store = load_sift_store_fast(need_ids)
    for img_id, (_, d) in sift_store.items():
        NKP_MAP[img_id] = len(d)
    print(f"Loaded RootSIFT for {len(need_ids):,} images in {time.time() - t0:.1f}s")

    by_q = {}
    for q, g in todo_pairs:
        by_q.setdefault(q, []).append(g)

    def _worker(item):
        q, gs = item
        qxy, qd16 = sift_store[q]
        qd32 = qd16.astype(np.float32)
        return [((q, g), match_pair_rich(qxy, qd32, sift_store[g][0], sift_store[g][1])) for g in gs]

    t1, done = time.time(), 0
    with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
        for res in ex.map(_worker, by_q.items(), chunksize=16):
            SIFT_CACHE.update(res)
            done += len(res)
            if done % 100000 < len(res):
                print(f"  SIFT progress: {done:,}/{len(todo_pairs):,} ({done / max(time.time() - t1, 1e-3):.0f} pairs/s)")
    with open(SIFT_CACHE_PATH, "wb") as fh:
        pickle.dump(SIFT_CACHE, fh)
    with open(NKP_CACHE_PATH, "wb") as fh:
        pickle.dump(NKP_MAP, fh)
    del sift_store
    gc.collect()
    print(f"Computed {len(todo_pairs):,} new SIFT pairs in {time.time() - t1:.1f}s -> cache updated.")

### 2.4 Feature engineering and the two-stage LightGBM model

* **Stage 1:** a LightGBM *identity model* gives P(candidate animal is correct); a LightGBM *new-animal model* gives P(query is `new_individual`) from query-level features. Both are trained out-of-fold, grouped by animal.
* **Stage 2:** the same models again, plus evidence from **confirmed same-animal links** between queries (cosine ≥ 0.80 or ≥ 14 SIFT inliers), via connected components (graph clustering). Raw neighbour-similarity features are deliberately not used, because they grow with the number of queries.
* **Feature pruning:** the identity features are ranked by LightGBM gain; the top `PRUNE_TOP_K` are kept for the ensemble if that costs at most 0.001 CV.

In [ ]:
def second_largest(s):
    s = s.dropna()
    return s.nlargest(2).iloc[-1] if len(s) >= 2 else np.nan

def add_rich_features(df, sift_cache, nkp_map, vp_map, sp_map):
    c = df.copy()
    n = len(c)
    inl_max = np.full(n, np.nan, dtype=np.float32)
    inl_2nd = np.full(n, np.nan, dtype=np.float32)
    good_max = np.full(n, np.nan, dtype=np.float32)
    inl_sum = np.full(n, np.nan, dtype=np.float32)
    inl_cnt8 = np.full(n, np.nan, dtype=np.float32)
    inl_cnt15 = np.full(n, np.nan, dtype=np.float32)
    ratio_max = np.full(n, np.nan, dtype=np.float32)
    sim_max = np.full(n, np.nan, dtype=np.float32)
    spread_max = np.full(n, np.nan, dtype=np.float32)
    nkp_g = np.full(n, np.nan, dtype=np.float32)

    n_comp_chk = np.full(n, np.nan, dtype=np.float32)
    comp_inl_max = np.full(n, np.nan, dtype=np.float32)
    comp_inl_norm = np.full(n, np.nan, dtype=np.float32)
    comp_spread = np.full(n, np.nan, dtype=np.float32)
    noncomp_inl = np.full(n, np.nan, dtype=np.float32)
    vrel = np.zeros(n, dtype=np.int8)

    for i, (q, imgs) in enumerate(zip(c["Query"].tolist(), c["BestImgs"].tolist())):
        vrel[i] = view_relation(vp_map[q], vp_map[imgs[0]])
        nkp_g[i] = nkp_map.get(imgs[0], np.nan)
        qs = get_side(vp_map[q])
        nkq_i = nkp_map.get(q, 0)

        vals, comp_vals, noncomp_vals = [], [], []
        for g in imgs:
            res = sift_cache.get((q, g))
            if res is None:
                continue
            vals.append(res)
            same = (get_side(vp_map[g]) == qs) if qs in "LR" else (vp_map[g] == vp_map[q] if qs == "O" else False)
            if same:
                comp_vals.append((res[1], res[1] / (min(nkq_i, nkp_map.get(g, 0)) + 1.0), res[4]))
            else:
                noncomp_vals.append(res[1])

        if vals:
            inls = sorted((v[1] for v in vals), reverse=True)
            inl_max[i] = inls[0]
            if len(inls) >= 2:
                inl_2nd[i] = inls[1]
            good_max[i] = max(v[0] for v in vals)
            inl_sum[i] = sum(inls)
            inl_cnt8[i] = sum(x >= 8 for x in inls)
            inl_cnt15[i] = sum(x >= 15 for x in inls)
            best_v = max(vals, key=lambda v: (v[1], v[3]))
            ratio_max[i], sim_max[i], spread_max[i] = best_v[2], best_v[3], best_v[4]
        if qs != "U":
            n_comp_chk[i] = len(comp_vals)
        if comp_vals:
            best_cv = max(comp_vals, key=lambda x: x[0])
            comp_inl_max[i], comp_inl_norm[i], comp_spread[i] = best_cv
        if noncomp_vals:
            noncomp_inl[i] = max(noncomp_vals)

    c["InlMax"], c["Inl2nd"], c["GoodMax"], c["InlSum"] = inl_max, inl_2nd, good_max, inl_sum
    c["InlCnt8"], c["InlCnt15"] = inl_cnt8, inl_cnt15
    c["InlRatio"], c["InlMeanSim"], c["InlSpread"] = ratio_max, sim_max, spread_max
    c["InlQuality"] = np.log1p(np.nan_to_num(inl_max, nan=0.0)) * np.nan_to_num(spread_max, nan=0.0)
    c["NChecked"] = c["BestImgs"].str.len()

    c["NCompChecked"], c["CompInlMax"], c["CompInlNorm"], c["CompSpread"] = n_comp_chk, comp_inl_max, comp_inl_norm, comp_spread
    c["NonCompInlMax"] = noncomp_inl
    c["NoCompView"] = np.where(c["NCompGal"] < 0, np.nan, (c["NCompGal"] == 0).astype(np.float32))
    c["LogNCompGal"] = np.log1p(c["NCompGal"].clip(lower=0))

    g_inl = c.groupby("QKey")["InlMax"]
    c["QInlMax"] = g_inl.transform("max")
    c["QInl2nd"] = g_inl.transform(second_largest)
    c["InlRank"] = g_inl.rank(ascending=False, method="min")
    c["InlFrac"] = c["InlMax"] / (c["QInlMax"] + 1.0)
    c["InlMargin"] = c["InlMax"] - np.where(c["InlMax"] >= c["QInlMax"], c["QInl2nd"], c["QInlMax"])
    c["QN12"] = (c["InlMax"] >= 12).groupby(c["QKey"]).transform("sum")
    c["QCompInlMax"] = c.groupby("QKey")["CompInlMax"].transform("max")

    nkq = c["Query"].map(nkp_map).astype(np.float32)
    c["InlNorm"] = c["InlMax"] / (np.minimum(nkq, nkp_g) + 1.0)
    c["LogNkpQ"] = np.log1p(nkq)
    c["LogNGal"] = np.log1p(c["NGal"])
    c["LogNEnc"] = np.log1p(c["NEncTotal"])

    # Gaps to the query's best candidate in each space
    c["CosGap"] = c["QTop1Cos"] - c["CosMax"]
    c["CSLSGap"] = c["QTop1CSLS"] - c["CSLSMax"]
    c["XCSLSGap"] = c["QTop1XCSLS"] - c["XCSLSMax"]
    c["JacGap"] = c["QTop1Jac"] - c["JacMax"]
    c["KDEGap"] = c["QTop1KDE"] - c["KDEScore"]
    c["TwoHopGap"] = c["QTop1TwoHop"] - c["TwoHopMax"]
    c["SameGap"] = c["CosMax"] - c["SameSideMax"]
    c["OppGap"] = c["CosMax"] - c["OppSideMax"]
    c["HasSameSide"] = c["SameSideMax"].notna().astype(np.int8)

    c["QRawTop1"] = c.groupby("QKey")["RawCosMax"].transform("max")
    c["QWPTop1"] = c.groupby("QKey")["WPCosMax"].transform("max")
    c["RawGap"] = c["QRawTop1"] - c["RawCosMax"]
    c["WPGap"] = c["QWPTop1"] - c["WPCosMax"]
    c["QKnnTop"] = c.groupby("QKey")["Knn15Frac"].transform("max")

    # Cross-metric consensus votes (CSLS, XQDA, Jaccard, KDE, two-hop, SIFT)
    c["Top1Votes"] = (
        (c["CSLSRank"] == 0).astype(np.int8) +
        (c["XCSLSRank"] == 0).astype(np.int8) +
        (c["JacRank"] == 0).astype(np.int8) +
        (c["KDERank"] == 0).astype(np.int8) +
        (c["TwoHopRank"] == 0).astype(np.int8) +
        (c["InlRank"] == 1).astype(np.int8)
    )
    c["QMaxVotes"] = c.groupby("QKey")["Top1Votes"].transform("max")

    q_r = c["Query"].map(ROW_MAP).to_numpy()
    b_r = c["BestImgs"].str[0].map(ROW_MAP).to_numpy()
    c["RawCosBest"] = (F_UNIT[q_r] * F_UNIT[b_r]).sum(axis=1)

    c["VRel"] = vrel
    c["SpeciesC"] = c["Query"].map(lambda q: SPECIES_LIST.index(sp_map[q])).astype(np.int8)
    c["QSide"] = c["Query"].map(lambda q: SIDE_CODE[get_side(vp_map[q])]).astype(np.int8)
    return c

t0 = time.time()
V_FEAT = add_rich_features(V_ALL, SIFT_CACHE, NKP_MAP, VP_MAP, SP_MAP)
T_FEAT = add_rich_features(CAND_TEST, SIFT_CACHE, NKP_MAP, VP_MAP, SP_MAP)
print(f"Features engineered in {time.time() - t0:.1f}s | V_FEAT: {V_FEAT.shape} | T_FEAT: {T_FEAT.shape}")

In [ ]:
ID_FEATS_1 = [
    "CosMax", "Cos2nd", "CosMean3", "CosGap",
    "CSLSMax", "CSLS2nd", "CSLSMean3", "CSLSGap", "CSLSRank",
    "XCosMax", "XCos2nd", "XCosMean3", "XCSLSMax", "XCSLS2nd", "XCSLSGap", "XCSLSRank",
    "JacMax", "JacGap", "JacRank", "KDEScore", "KDEShare", "KDEGap", "KDERank",
    "TwoHopMax", "TwoHopGap", "TwoHopRank",
    "CentroidCos", "CosZ", "XCosZ", "GalHub", "QHub", "QXHub",
    "Knn15Frac", "Cos2ndEnc", "LogNEnc", "NEncStrong",
    "SameSideMax", "SameSideXMax", "SameSideCSLS", "SameGap", "OppSideMax", "OppGap",
    "OtherViewMax", "NSides", "HasSameSide", "IdRank",
    "Top1Votes", "QMaxVotes",
    "QTop1Cos", "QGap12Cos", "QTop1CSLS", "QGap12CSLS", "QTop1XCSLS", "QGap12XCSLS",
    "QTop1Jac", "QGap12Jac", "QTop1KDE", "QGap12KDE", "QTop1TwoHop", "QGap12TwoHop",
    "LogNGal", "RawCosBest", "RawCosMax", "RawRank", "RawGap", "WPCosMax", "WPRank", "WPGap", "WFCosMax",
    "InlMax", "Inl2nd", "GoodMax", "InlSum", "InlCnt8", "InlCnt15", "InlRatio", "InlMeanSim", "InlSpread", "InlQuality",
    "NChecked", "InlRank", "QInlMax", "InlFrac", "InlNorm", "InlMargin", "QN12",
    "NCompChecked", "CompInlMax", "CompInlNorm", "CompSpread", "NonCompInlMax",
    "NoCompView", "LogNCompGal", "LogNkpQ", "VRel", "SpeciesC", "QSide",
]

Q_BASE_1 = [
    "QTop1Cos", "QGap12Cos", "QTop1CSLS", "QGap12CSLS", "QTop1XCSLS", "QGap12XCSLS",
    "QTop1Jac", "QGap12Jac", "QTop1KDE", "QGap12KDE", "QTop1TwoHop", "QGap12TwoHop",
    "QMaxVotes", "QHub", "QXHub", "QInlMax", "QInl2nd", "QN12", "QKnnTop", "LogNkpQ",
    "SpeciesC", "QSide", "QRawTop1", "QWPTop1", "QCompInlMax",
]
Q_FEATS_1 = Q_BASE_1 + ["PMax", "P2nd", "PSum", "PGap12", "PEntropy"]

LGB_ID_PARAMS = dict(
    n_estimators=750, learning_rate=0.018, num_leaves=15, min_child_samples=80,
    subsample=0.8, subsample_freq=1, colsample_bytree=0.72, reg_alpha=1.5, reg_lambda=5.0,
    verbose=-1, random_state=SEED, n_jobs=N_WORKERS,
)
LGB_NEW_PARAMS = dict(
    n_estimators=500, learning_rate=0.018, num_leaves=15, min_child_samples=50,
    subsample=0.8, subsample_freq=1, colsample_bytree=0.75, reg_alpha=1.5, reg_lambda=5.0,
    verbose=-1, random_state=SEED, n_jobs=N_WORKERS,
)

def build_query_table(df, base_cols):
    g = df.groupby("QKey", sort=False)
    Q = g[base_cols].first()
    Q["PMax"] = g["P"].max()
    Q["P2nd"] = g["P"].apply(second_largest).fillna(0.0)
    Q["PSum"] = g["P"].sum()
    Q["PGap12"] = Q["PMax"] - Q["P2nd"]
    def _ent(p):
        s = p.sum()
        if s <= 1e-9:
            return 0.0
        q = np.clip(p / s, 1e-9, 1.0)
        return float(-(q * np.log(q)).sum())
    Q["PEntropy"] = g["P"].apply(_ent)
    return Q

def oof_predict_lgb(X, y, groups, params, n_splits=5):
    out = np.zeros(len(X), dtype=np.float64)
    for tr, va in GroupKFold(n_splits).split(X, y, groups):
        m = lgb.LGBMClassifier(**params).fit(X.iloc[tr], y[tr])
        out[va] = m.predict_proba(X.iloc[va])[:, 1]
    return out

def eval_stage_map5(df, p_new_series, bias=0.0):
    preds, truth = {}, {}
    for qk, g in df.groupby("QKey", sort=False):
        truth[qk] = g["Truth"].iloc[0]
        p0 = float(np.clip(p_new_series[qk], 1e-6, 1.0 - 1e-6))
        pn = float(expit(logit(p0) + bias))
        pk = g["P"].to_numpy()
        sk = (1.0 - pn) * (pk / max(pk.sum(), 1.0))
        labs = g["Identity"].tolist() + [NEW]
        scores = np.r_[sk, pn]
        preds[qk] = [labs[i] for i in np.argsort(-scores)[:5]]
    return map5_score(truth, preds)

t0 = time.time()
y_id = (V_FEAT["Identity"] == V_FEAT["Truth"]).to_numpy(int)
id_groups = V_FEAT["Query"].map(LAB_MAP).to_numpy()

V_FEAT["P"] = oof_predict_lgb(V_FEAT[ID_FEATS_1], y_id, id_groups, LGB_ID_PARAMS)
m_id_1 = lgb.LGBMClassifier(**LGB_ID_PARAMS).fit(V_FEAT[ID_FEATS_1], y_id)
T_FEAT["P"] = m_id_1.predict_proba(T_FEAT[ID_FEATS_1])[:, 1]

QV1 = build_query_table(V_FEAT, Q_BASE_1)
QT1 = build_query_table(T_FEAT, Q_BASE_1)
y_new = (V_FEAT.groupby("QKey")["Truth"].first().loc[QV1.index] == NEW).to_numpy(int)
q_groups = QV1.index.str.split(":").str[1].map(LAB_MAP).to_numpy()

p_new_v1 = pd.Series(oof_predict_lgb(QV1[Q_FEATS_1], y_new, q_groups, LGB_NEW_PARAMS), index=QV1.index)
m_new_1 = lgb.LGBMClassifier(**LGB_NEW_PARAMS).fit(QV1[Q_FEATS_1], y_new)
p_new_t1 = pd.Series(m_new_1.predict_proba(QT1[Q_FEATS_1])[:, 1], index=QT1.index)

s1_map5 = eval_stage_map5(V_FEAT, p_new_v1, bias=0.0)
print(f"Stage 1 (metric learning + KDE/two-hop + SIFT) CV MAP@5: {s1_map5:.5f} ({time.time() - t0:.1f}s)")

# ---- Stage 2: Density-Invariant Confirmed-Link Propagation (NO raw NbrCos!) ----
def add_confirmed_graph_features(df, p_new_map, q_nbrs, prefix, sift_cache, cos_thr=0.80, inl_thr=14):
    """
    Only propagates identity evidence across HIGH-PRECISION confirmed same-animal links.
    Deliberately excludes raw NbrCos/CompSize so query-set size differences (CV vs Test)
    cannot cause distribution shift.
    """
    P_dict = {(q, ident): p for q, ident, p in zip(df["Query"], df["Identity"], df["P"])}
    q_list = df["Query"].unique().tolist()

    G = nx.Graph()
    G.add_nodes_from(q_list)
    conf_links = {q: [] for q in q_list}

    for q in q_list:
        for q2, cs in q_nbrs.get(q, []):
            key = (q, q2) if q < q2 else (q2, q)
            s_res = sift_cache.get(key)
            inl = s_res[1] if s_res is not None else 0
            if cs >= cos_thr or inl >= inl_thr or (cs >= 0.72 and inl >= 9):
                w = max(cs, min(1.0, inl / 20.0))
                G.add_edge(q, q2, weight=w)
                conf_links[q].append((q2, w))

    comp_members, comp_mean_new = {}, {}
    for comp in nx.connected_components(G):
        c_list = list(comp)
        for q in c_list:
            others = [x for x in c_list if x != q]
            comp_members[q] = others
            comp_mean_new[q] = float(np.mean([p_new_map.get(prefix + x, np.nan) for x in others])) if others else np.nan

    out = df.copy()
    conf_nbr_p, conf_comp_max_p = [], []
    for q, ident in zip(out["Query"], out["Identity"]):
        lk = conf_links[q]
        conf_nbr_p.append(max((w * P_dict.get((q2, ident), 0.0) for q2, w in lk), default=0.0))
        others = comp_members[q]
        conf_comp_max_p.append(max((P_dict.get((q2, ident), 0.0) for q2 in others), default=0.0) if others else 0.0)

    out["ConfNbrP"] = conf_nbr_p
    out["ConfCompMaxP"] = conf_comp_max_p
    out["Q_ConfCompMaxP"] = out.groupby("QKey")["ConfCompMaxP"].transform("max")
    out["Q_ConfMeanNew"] = out["Query"].map(comp_mean_new)
    return out

v_parts = []
for f in FOLDS:
    part = V_FEAT[V_FEAT["Fold"] == f["fold"]]
    v_parts.append(add_confirmed_graph_features(part, p_new_v1.to_dict(), FOLD_QNBRS[f["fold"]], f"{f['fold']}:", SIFT_CACHE))
V2 = pd.concat(v_parts, ignore_index=True)
T2 = add_confirmed_graph_features(T_FEAT, p_new_t1.to_dict(), TEST_QNBRS, "", SIFT_CACHE)

ID_FEATS_2 = ID_FEATS_1 + ["ConfNbrP", "ConfCompMaxP"]
Q_BASE_2 = Q_BASE_1 + ["Q_ConfCompMaxP", "Q_ConfMeanNew"]
Q_FEATS_2 = Q_BASE_2 + ["PMax", "P2nd", "PSum", "PGap12", "PEntropy"]

y_id_2 = (V2["Identity"] == V2["Truth"]).to_numpy(int)
id_groups_2 = V2["Query"].map(LAB_MAP).to_numpy()

V2["P"] = oof_predict_lgb(V2[ID_FEATS_2], y_id_2, id_groups_2, LGB_ID_PARAMS)
m_id_2 = lgb.LGBMClassifier(**LGB_ID_PARAMS).fit(V2[ID_FEATS_2], y_id_2)
T2["P"] = m_id_2.predict_proba(T2[ID_FEATS_2])[:, 1]

QV2 = build_query_table(V2, Q_BASE_2)
QT2 = build_query_table(T2, Q_BASE_2)
y_new_2 = (V2.groupby("QKey")["Truth"].first().loc[QV2.index] == NEW).to_numpy(int)
q_groups_2 = QV2.index.str.split(":").str[1].map(LAB_MAP).to_numpy()

p_new_v2 = pd.Series(oof_predict_lgb(QV2[Q_FEATS_2], y_new_2, q_groups_2, LGB_NEW_PARAMS), index=QV2.index)
m_new_2 = lgb.LGBMClassifier(**LGB_NEW_PARAMS).fit(QV2[Q_FEATS_2], y_new_2)
p_new_t2 = pd.Series(m_new_2.predict_proba(QT2[Q_FEATS_2])[:, 1], index=QT2.index)

s2_map5 = eval_stage_map5(V2, p_new_v2, bias=0.0)
print(f"Stage 2 (+Confirmed Graph) CV MAP@5: {s2_map5:.5f} (delta: {s2_map5 - s1_map5:+.5f})")
gain = pd.Series(m_id_2.booster_.feature_importance("gain"), ID_FEATS_2).sort_values(ascending=False)
print("\nTop 15 identity-model features (share of total gain):")
print((gain / gain.sum()).head(15).round(3).to_string())

# ---- feature pruning: keep the PRUNE_TOP_K most useful identity features if CV does not drop ----
ID_FEATS_ENS = ID_FEATS_2
if PRUNE_TOP_K and PRUNE_TOP_K < len(ID_FEATS_2):
    keep = list(gain.index[:PRUNE_TOP_K])
    V2p = V2.copy()
    V2p["P"] = oof_predict_lgb(V2p[keep], y_id_2, id_groups_2, LGB_ID_PARAMS)
    s2p_map5 = eval_stage_map5(V2p, p_new_v2, bias=0.0)
    print(f"\nStage 2 with the top {PRUNE_TOP_K} identity features: CV MAP@5 {s2p_map5:.5f} "
          f"(all {len(ID_FEATS_2)}: {s2_map5:.5f})")
    if s2p_map5 >= s2_map5 - 0.001:
        ID_FEATS_ENS = keep
        print(f"-> the ensemble uses the {len(keep)} pruned features")
    else:
        print("-> pruning costs CV; the ensemble keeps all features")
    del V2p

### 2.5 Evaluation helpers

* **Test new-animal share:** estimated per species by prior-shift EM (Saerens et al., 2002) on the stage-2 new-animal model's test probabilities. The validation folds contain more new animals than test, so…
* **re-weighted CV:** each species' `new_individual` queries are re-weighted to the estimated test share, and species are combined in their test proportions. Model 2's ensemble is tuned and reported on this number.
* **New-animal shift:** one shift per species moves `new_individual` up or down; it is chosen on the re-weighted CV.

In [ ]:
def build_rank_table(df, q_cols):
    """each query's candidate rows plus one new_individual row (query-level features copied)"""
    R = df.copy()
    R["IsNewRow"] = 0
    q_first = df.groupby("QKey", sort=False).first().reset_index()
    n_rows = pd.DataFrame({"QKey": q_first["QKey"], "Query": q_first["Query"], "Identity": NEW,
                           "IsNewRow": 1, "SpeciesC": q_first["SpeciesC"]})
    if "Truth" in q_first.columns:
        n_rows["Truth"] = q_first["Truth"].values
    if "Fold" in q_first.columns:
        n_rows["Fold"] = q_first["Fold"].values
    for col in q_cols:
        if col not in n_rows.columns:
            n_rows[col] = q_first[col].values
    R = pd.concat([R, n_rows], ignore_index=True)
    return R.sort_values(["QKey", "IsNewRow"], kind="stable").reset_index(drop=True)


RV = build_rank_table(V2, Q_BASE_2)
RT = build_rank_table(T2, Q_BASE_2)
y_rv = (RV["Identity"] == RV["Truth"]).to_numpy(int)
rv_groups = RV["Query"].map(LAB_MAP).to_numpy()
rv_qcode, rv_qkeys = pd.factorize(RV["QKey"])
rt_qcode, rt_qkeys = pd.factorize(RT["QKey"])
rv_is_new = (RV["IsNewRow"].to_numpy() == 1)
rt_is_new = (RT["IsNewRow"].to_numpy() == 1)


def clf_to_row_probs(df_cand, p_new_series, R_target):
    """identity probabilities x (1 - P(new)) for candidate rows, P(new) for the new_individual row"""
    p_map = df_cand.set_index(["QKey", "Identity"])["P"]
    code = pd.factorize(R_target["QKey"])[0]
    is_new = (R_target["IsNewRow"].to_numpy() == 1)
    p = pd.MultiIndex.from_frame(R_target[["QKey", "Identity"]]).map(p_map).to_numpy(dtype=np.float64)
    p = np.nan_to_num(p, nan=0.0)
    pn = np.clip(R_target["QKey"].map(p_new_series).to_numpy(dtype=np.float64), 1e-6, 1.0 - 1e-6)
    p_sum = pd.Series(p * (~is_new)).groupby(code).transform("sum").to_numpy()
    raw_p = np.where(is_new, pn, (1.0 - pn) * (p / np.maximum(p_sum, 1.0)))
    tot = pd.Series(raw_p).groupby(code).transform("sum").to_numpy()
    return np.clip(raw_p / np.maximum(tot, 1e-12), 1e-9, 1.0)


QUERY_TRUTH = RV.groupby("QKey", sort=False)["Truth"].first()
QUERY_SPECIES = RV.groupby("QKey", sort=False)["SpeciesC"].first().map(lambda i: SPECIES_LIST[int(i)])
TEST_SPECIES_SHARE = test["species"].value_counts(normalize=True).to_dict()


def rr_from_row_probs(R_df, probs):
    """reciprocal rank of the true label per query (0 if it is not in the top 5)"""
    code, keys = pd.factorize(R_df["QKey"])
    order = np.lexsort((-probs, code))
    g = code[order]
    pos = np.arange(len(g)) - np.searchsorted(g, g)
    hit = (R_df["Identity"].to_numpy() == R_df["Truth"].to_numpy())[order] & (pos < 5)
    per_q = np.zeros(len(keys))
    per_q[g[hit]] = 1.0 / (pos[hit] + 1)
    return pd.Series(per_q, index=keys)


def weighted_cv(rr, rates, species=None):
    """MAP@5 with each species' new_individual queries re-weighted to its estimated test share;
    species combined in their test proportions (or one species only)."""
    sp = QUERY_SPECIES.reindex(rr.index)
    is_new = (QUERY_TRUTH.reindex(rr.index) == NEW)
    out = 0.0
    for s in ([species] if species else SPECIES_LIST):
        m = (sp == s)
        sc = rates[s] * rr[m & is_new].mean() + (1.0 - rates[s]) * rr[m & ~is_new].mean()
        out += sc if species else TEST_SPECIES_SHARE[s] * sc
    return float(out)


def saerens_em(p_test_new, val_prior, n_iter=200):
    """prior-shift EM: the new-animal share in test consistent with the model's test probabilities"""
    p = float(val_prior)
    for _ in range(n_iter):
        w = (p / val_prior) / ((1.0 - p) / (1.0 - val_prior))
        p = float(np.mean((p_test_new * w) / (p_test_new * w + 1.0 - p_test_new)))
    return p


EST_NEW = {}
for i, s in enumerate(SPECIES_LIST):
    val_prior = float(y_new_2[(QV2["SpeciesC"] == i).to_numpy()].mean())
    est = saerens_em(p_new_t2[(QT2["SpeciesC"] == i).to_numpy()].to_numpy(), val_prior)
    EST_NEW[s] = float(np.clip(est, 0.05, 0.70))  # safety guard only
    print(f"{s:8s}: new_individual share  validation {val_prior:.1%} -> estimated test {est:.1%}")


def shift_new(R_df, probs, shift):
    """multiply the new_individual row by exp(shift) and renormalise per query"""
    adj = probs * np.where(R_df["IsNewRow"].to_numpy() == 1, np.exp(shift), 1.0)
    code = pd.factorize(R_df["QKey"])[0]
    return adj / pd.Series(adj).groupby(code).transform("sum").to_numpy()


def tune_new_shift(P_oof, mask=None):
    """per species: the new_individual shift that maximises the re-weighted CV"""
    shifts = {}
    for i, s in enumerate(SPECIES_LIST):
        rows = (RV["SpeciesC"].to_numpy() == i)
        if mask is not None:
            rows = rows & mask
        R_s = RV[rows]
        best = max((weighted_cv(rr_from_row_probs(R_s, shift_new(R_s, P_oof[rows], b)), EST_NEW, s), b)
                   for b in np.round(np.linspace(-1.5, 1.5, 31), 2))
        shifts[s] = float(best[1])
    return shifts


def apply_shifts(R_df, probs, shifts):
    out = probs.copy()
    for i, s in enumerate(SPECIES_LIST):
        rows = (R_df["SpeciesC"].to_numpy() == i)
        out[rows] = shift_new(R_df[rows], probs[rows], shifts[s])
    return out


def top5_preds(R_df, probs):
    code = pd.factorize(R_df["QKey"])[0]
    order = np.lexsort((-probs, code))
    g = code[order]
    pos = np.arange(len(g)) - np.searchsorted(g, g)
    top = R_df.iloc[order[pos < 5]][["Query", "Identity"]]
    return top.groupby("Query", sort=False)["Identity"].apply(list).to_dict()


RESULTS = {}  # model name -> row of the comparison table

### 2.6 Ensemble and submission

Five members, all trained out-of-fold (grouped by animal) on the same features:
* `cb_clf` — CatBoost identity + new-animal models (2 seeds)
* `mlp_clf` — scikit-learn MLP (3 seeds)
* `lgb_rank` — LightGBM lambdarank over each query's candidates + a `new_individual` row (3 seeds)
* `lgb_rank_sp` — the same ranker trained separately per species (2 seeds)
* `xgb_rank` — XGBoost ranker (2 seeds)

Members are blended with weights that minimise out-of-fold log-loss. A **held-out check** fits the weights on folds 0–1 and scores them on folds 2–4 (and the reverse), so the reported blend score is not tuned on the data it is scored on.

In [ ]:
RANK_FEATS = list(dict.fromkeys(ID_FEATS_ENS + [c for c in Q_BASE_2 if c not in ID_FEATS_ENS] + ["IsNewRow"]))


def fit_two_stage_clf(name, model_fn_seed, seeds=(SEED,), per_species=False):
    t0 = time.time()
    vc, tc = V2.copy(), T2.copy()
    oof_p_tot = np.zeros(len(vc), dtype=np.float64)
    test_p_tot = np.zeros(len(tc), dtype=np.float64)
    sp_slices = range(len(SPECIES_LIST)) if per_species else [None]

    for sd in seeds:
        oof_p = np.zeros(len(vc), dtype=np.float64)
        test_p = np.zeros(len(tc), dtype=np.float64)
        for sp_idx in sp_slices:
            vm = (vc["SpeciesC"] == sp_idx).to_numpy() if sp_idx is not None else np.ones(len(vc), bool)
            tm = (tc["SpeciesC"] == sp_idx).to_numpy() if sp_idx is not None else np.ones(len(tc), bool)
            X_v, y_v, g_v = vc.loc[vm, ID_FEATS_ENS], y_id_2[vm], id_groups_2[vm]
            sub_oof = np.zeros(vm.sum(), dtype=np.float64)
            for tr, va in GroupKFold(5).split(X_v, y_v, g_v):
                sub_oof[va] = model_fn_seed(sd).fit(X_v.iloc[tr], y_v[tr]).predict_proba(X_v.iloc[va])[:, 1]
            oof_p[vm] = sub_oof
            test_p[tm] = model_fn_seed(sd).fit(X_v, y_v).predict_proba(tc.loc[tm, ID_FEATS_ENS])[:, 1]
        oof_p_tot += oof_p / len(seeds)
        test_p_tot += test_p / len(seeds)

    vc["P"], tc["P"] = oof_p_tot, test_p_tot
    qv, qt = build_query_table(vc, Q_BASE_2), build_query_table(tc, Q_BASE_2)
    oof_n_tot = np.zeros(len(qv), dtype=np.float64)
    test_n_tot = np.zeros(len(qt), dtype=np.float64)

    for sd in seeds:
        oof_n = np.zeros(len(qv), dtype=np.float64)
        test_n = np.zeros(len(qt), dtype=np.float64)
        for sp_idx in sp_slices:
            vm = (qv["SpeciesC"] == sp_idx).to_numpy() if sp_idx is not None else np.ones(len(qv), bool)
            tm = (qt["SpeciesC"] == sp_idx).to_numpy() if sp_idx is not None else np.ones(len(qt), bool)
            X_v, y_v, g_v = qv.loc[vm, Q_FEATS_2], y_new_2[vm], q_groups_2[vm]
            sub_oof = np.zeros(vm.sum(), dtype=np.float64)
            for tr, va in GroupKFold(5).split(X_v, y_v, g_v):
                sub_oof[va] = model_fn_seed(sd).fit(X_v.iloc[tr], y_v[tr]).predict_proba(X_v.iloc[va])[:, 1]
            oof_n[vm] = sub_oof
            test_n[tm] = model_fn_seed(sd).fit(X_v, y_v).predict_proba(qt.loc[tm, Q_FEATS_2])[:, 1]
        oof_n_tot += oof_n / len(seeds)
        test_n_tot += test_n / len(seeds)

    print(f"  Trained {name:12s} ({len(seeds)} seeds) in {time.time() - t0:.1f}s")
    return clf_to_row_probs(vc, pd.Series(oof_n_tot, index=qv.index), RV), clf_to_row_probs(tc, pd.Series(test_n_tot, index=qt.index), RT)

def softmax_by_query(scores, qcode, is_new, temp=1.0, new_shift=0.0):
    s = scores / max(temp, 1e-3) + new_shift * is_new
    s = s - pd.Series(s).groupby(qcode).transform("max").to_numpy()
    e = np.exp(np.clip(s, -40, 0))
    denom = pd.Series(e).groupby(qcode).transform("sum").to_numpy()
    return np.clip(e / np.maximum(denom, 1e-12), 1e-9, 1.0)

def calibrate_ranker_probs(oof_scores, test_scores):
    pos_mask = (y_rv == 1)
    def loss_fn(params):
        temp, shift = params
        p = softmax_by_query(oof_scores, rv_qcode, rv_is_new, temp=np.exp(temp), new_shift=shift)
        return -float(np.mean(np.log(p[pos_mask])))
    res = minimize(loss_fn, x0=[0.0, 0.0], method="Nelder-Mead")
    t_opt, b_opt = float(np.exp(res.x[0])), float(res.x[1])
    return (
        softmax_by_query(oof_scores, rv_qcode, rv_is_new, temp=t_opt, new_shift=b_opt),
        softmax_by_query(test_scores, rt_qcode, rt_is_new, temp=t_opt, new_shift=b_opt),
    )

def fit_ranker_member(name, kind, seeds=(SEED,), per_species=False):
    t0 = time.time()
    oof_tot = np.zeros(len(RV), dtype=np.float64)
    test_tot = np.zeros(len(RT), dtype=np.float64)
    sp_slices = range(len(SPECIES_LIST)) if per_species else [None]

    for sd in seeds:
        oof = np.zeros(len(RV), dtype=np.float64)
        tst = np.zeros(len(RT), dtype=np.float64)
        for sp_idx in sp_slices:
            vm = (RV["SpeciesC"] == sp_idx).to_numpy() if sp_idx is not None else np.ones(len(RV), bool)
            tm = (RT["SpeciesC"] == sp_idx).to_numpy() if sp_idx is not None else np.ones(len(RT), bool)
            R_sub = RV.loc[vm]
            y_sub = y_rv[vm]
            g_sub = rv_groups[vm]
            sub_oof = np.zeros(len(R_sub), dtype=np.float64)

            def _fit_one(R_df, y_arr):
                if kind == "lgb":
                    return lgb.LGBMRanker(
                        objective="lambdarank", n_estimators=600, learning_rate=0.025,
                        num_leaves=31, min_child_samples=50, subsample=0.8, subsample_freq=1,
                        colsample_bytree=0.75, reg_alpha=1.5, reg_lambda=5.0,
                        lambdarank_truncation_level=8, verbose=-1, random_state=sd, n_jobs=N_WORKERS
                    ).fit(R_df[RANK_FEATS], y_arr, group=R_df.groupby("QKey", sort=False).size().to_numpy())
                else:
                    return xgb.XGBRanker(
                        objective="rank:ndcg", eval_metric="map@5", tree_method="hist",
                        n_estimators=500, learning_rate=0.035, max_depth=5, min_child_weight=10,
                        subsample=0.8, colsample_bytree=0.75, reg_alpha=1.5, reg_lambda=5.0,
                        random_state=sd, n_jobs=N_WORKERS
                    ).fit(R_df[RANK_FEATS], y_arr, qid=pd.factorize(R_df["QKey"])[0])

            for tr, va in GroupKFold(5).split(R_sub, y_sub, g_sub):
                m = _fit_one(R_sub.iloc[tr], y_sub[tr])
                sub_oof[va] = m.predict(R_sub.iloc[va][RANK_FEATS])
            oof[vm] = sub_oof
            m_full = _fit_one(R_sub, y_sub)
            tst[tm] = m_full.predict(RT.loc[tm, RANK_FEATS])
        oof_tot += oof / len(seeds)
        test_tot += tst / len(seeds)

    def _qstd(x, qc):
        s = pd.Series(x).groupby(qc)
        return (x - s.transform("mean").to_numpy()) / (s.transform("std").to_numpy() + 1e-6)
    p_oof, p_test = calibrate_ranker_probs(_qstd(oof_tot, rv_qcode), _qstd(test_tot, rt_qcode))
    print(f"  Trained {name:12s} ({len(seeds)} seeds) in {time.time() - t0:.1f}s")
    return p_oof, p_test


print("Training the 5-member ensemble...")
MEMBERS_OOF, MEMBERS_TEST = {}, {}
MEMBERS_OOF["cb_clf"], MEMBERS_TEST["cb_clf"] = fit_two_stage_clf(
    "cb_clf",
    lambda sd: CatBoostClassifier(iterations=700, learning_rate=0.035, depth=6, l2_leaf_reg=6.0,
                                  verbose=0, random_seed=sd, thread_count=N_WORKERS),
    seeds=(SEED, SEED + 1))
MEMBERS_OOF["mlp_clf"], MEMBERS_TEST["mlp_clf"] = fit_two_stage_clf(
    "mlp_clf",
    lambda sd: make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler(),
                             MLPClassifier(hidden_layer_sizes=(96, 48), alpha=5e-2, max_iter=180,
                                           early_stopping=True, random_state=sd)),
    seeds=(SEED, SEED + 1, SEED + 2))
MEMBERS_OOF["lgb_rank"], MEMBERS_TEST["lgb_rank"] = fit_ranker_member("lgb_rank", "lgb", seeds=(SEED, SEED + 1, SEED + 2))
MEMBERS_OOF["lgb_rank_sp"], MEMBERS_TEST["lgb_rank_sp"] = fit_ranker_member("lgb_rank_sp", "lgb", seeds=(SEED, SEED + 1), per_species=True)
MEMBERS_OOF["xgb_rank"], MEMBERS_TEST["xgb_rank"] = fit_ranker_member("xgb_rank", "xgb", seeds=(SEED, SEED + 1))
with open(WORK_DIR / "m2_ensemble_preds.pkl", "wb") as fh:
    pickle.dump({"oof": MEMBERS_OOF, "test": MEMBERS_TEST}, fh)

print("\nEach member alone, CV MAP@5 (validation mix):")
for name, p_oof in MEMBERS_OOF.items():
    print(f"  {name:12s}: {rr_from_row_probs(RV, p_oof).mean():.5f}")

m_names = list(MEMBERS_OOF.keys())
P_oof_mat = np.column_stack([MEMBERS_OOF[n] for n in m_names])
P_test_mat = np.column_stack([MEMBERS_TEST[n] for n in m_names])
pos_idx = (y_rv == 1)


def fit_blend_weights(rows_mask):
    """simplex weights that minimise out-of-fold log-loss of the true label (rows_mask: queries used)"""
    pos = pos_idx & rows_mask
    def loss(raw_w):
        w = np.exp(raw_w - raw_w.max())
        w /= w.sum()
        return -float(np.mean(np.log(np.clip(P_oof_mat[pos] @ w, 1e-9, 1.0))))
    res = minimize(loss, x0=np.zeros(len(m_names)), method="BFGS")
    w = np.exp(res.x - res.x.max())
    return w / w.sum()


# held-out check: weights and new_individual shift fitted on some folds, scored on the others
fold_arr = RV["Fold"].to_numpy()
held = []
for A, B in (([0, 1], [2, 3, 4]), ([2, 3, 4], [0, 1])):
    mA, mB = np.isin(fold_arr, A), np.isin(fold_arr, B)
    wA = fit_blend_weights(mA)
    shA = tune_new_shift(P_oof_mat @ wA, mask=mA)
    pB = apply_shifts(RV, P_oof_mat @ wA, shA)[mB]
    held.append(weighted_cv(rr_from_row_probs(RV[mB], pB), EST_NEW))
M3_HELD = float(np.mean(held))
print(f"\nHeld-out blend CV (re-weighted to test mix): {M3_HELD:.5f}  (folds 2-4: {held[0]:.5f}, folds 0-1: {held[1]:.5f})")

W_OPT = fit_blend_weights(np.ones(len(RV), dtype=bool))
print("Blend weights (all folds):", {n: round(float(w), 3) for n, w in zip(m_names, W_OPT)})
P_ens_oof, P_ens_test = P_oof_mat @ W_OPT, P_test_mat @ W_OPT
M3_SHIFTS = tune_new_shift(P_ens_oof)
P3_oof_s = apply_shifts(RV, P_ens_oof, M3_SHIFTS)
P3_test_s = apply_shifts(RT, P_ens_test, M3_SHIFTS)
m3_rr = rr_from_row_probs(RV, P3_oof_s)
print(f"Model 2: new_individual shifts {M3_SHIFTS}")
print(f"  CV MAP@5 (validation mix, no shift) {rr_from_row_probs(RV, P_ens_oof).mean():.5f} | "
      f"re-weighted to estimated test mix {weighted_cv(m3_rr, EST_NEW):.5f}")
M3_PREDS = top5_preds(RT, P3_test_s)
check_and_save_submission(M3_PREDS, filename="model2_xqda_kde_ensemble.csv")
RESULTS["Model 2"] = {"what": "5-member ensemble (CatBoost, MLP, LightGBM rankers, XGBoost ranker)",
                      "cv": rr_from_row_probs(RV, P_ens_oof).mean(), "cv_test_mix": weighted_cv(m3_rr, EST_NEW),
                      "held_out": M3_HELD, "csv": "model2_xqda_kde_ensemble.csv"}
print(f"Total runtime so far: {(time.time() - T_START) / 60.0:.1f} minutes")

In [ ]:
%reset -f
# Model 2's CSV is on disk; clear its objects to free memory so the next model starts clean.
import gc
gc.collect()

## Model 3 — V8 fusion pipeline from `projectwild3` (Kaggle public 0.85677)

The scripts `Common.py`, `P1Validation.py`, `P2Global.py`, `P3Sift.py`, `Stage2.py` and the V8 `P4Fusion.py` (`Backup_V8`), written out as cells. The code is unchanged except: file paths come from the same path search as Model 1, hand-off files go to `Work/Model3_V8/`, the CSV is written next to the notebook, and two functions are renamed so they don't clash (`Main` → `GlobalMain` / `FusionMain`, `Run` → `SiftRun`).

* **Validation (P1):** 4 folds; per fold 20 % of each species' animals are hidden (`new_individual`), and for the other animals whole **encounters** (bursts with whitened cosine > 0.8) are held out as queries.
* **Retrieval (P2):** per-species PCA whitening (512) + LDA (128) fitted on gallery labels only; 50 candidate animals per query (40 by the main cosine + 10 by raw MiewID cosine).
* **SIFT (P3):** RootSIFT, ratio test 0.8, one-to-one matching, similarity-transform RANSAC (15 px) on the provided keypoints; all photos (up to 40) of the 3 best candidates and 3 photos of the others.
* **Fusion (P4):** two-stage LightGBM (identity model + new-animal model, out-of-fold, grouped by animal); stage 2 adds evidence from neighbouring queries, with test queries split into fold-sized groups.
* **Ensemble (V8):** 9 members — `lgb` (two-stage LightGBM), `lgbrank` (LightGBM lambdarank, 3 seeds), `xgbrank` (XGBoost rank:map), `cbrank` (CatBoost YetiRank), and two-stage CatBoost, HistGradientBoosting, random forest, logistic regression and MLP. Blend weights and each member's `new_individual` shift are tuned by coordinate ascent on out-of-fold MAP@5, re-weighted to the test new-animal share estimated by prior-shift EM. The weights are tuned on the same out-of-fold scores they are reported on, so that CV is optimistic.

Output: `model3_v8_fusion_ensemble.csv`.

### 3.1 Setup and shared helpers (`Common.py`)

In [ ]:
import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_v, "1")

import pickle
import time
import warnings
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import cv2
import lightgbm as lgb
import numpy as np
import pandas as pd
import xgboost as xgb
from catboost import CatBoostClassifier, CatBoostRanker
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
cv2.setNumThreads(1)
T_START = time.time()
NEW = "new_individual"


def resolve_paths(base_override=None):
    candidates = []
    if base_override:
        candidates.append(Path(base_override))
    candidates += [
        Path.cwd(),
        Path.cwd() / "reid_student_data_main",
        Path.cwd() / "projectwild3",
        Path.cwd() / "projectwild3" / "reid_student_data_main",
        Path("/kaggle/input"),
    ]
    # Also search immediate subdirs of /kaggle/input if present
    ki = Path("/kaggle/input")
    if ki.exists():
        for p in ki.rglob("train.csv"):
            candidates.insert(0, p.parent)

    data_dir = None
    for c in candidates:
        if (c / "train.csv").exists() and (c / "miewid_features.npy").exists():
            data_dir = c
            break
        if (c / "reid_student_data_main" / "train.csv").exists():
            data_dir = c / "reid_student_data_main"
            break
    if data_dir is None:
        raise FileNotFoundError(
            "Could not locate train.csv and miewid_features.npy. "
            "Set BASE_DIR = '/path/to/your/folder' above."
        )

    def find_sift(name):
        for p in [data_dir / name, data_dir.parent / name, Path.cwd() / name]:
            if p.exists():
                return p
        # Search upward/siblings on Kaggle
        for p in data_dir.parent.rglob(name):
            return p
        raise FileNotFoundError(f"Could not locate {name}")

    work_dir = Path("/kaggle/working/Work") if Path("/kaggle/working").exists() else (Path.cwd() / "Work")
    work_dir.mkdir(parents=True, exist_ok=True)
    out_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
    return data_dir, find_sift("sift_features_train.npz"), find_sift("sift_features_test.npz"), work_dir, out_dir


DATA, SIFT_TRAIN_PATH, SIFT_TEST_PATH, WORK_ROOT, OUT_DIR = resolve_paths(os.environ.get("REID_BASE"))
SIFT_SRC = {"tr": SIFT_TRAIN_PATH, "te": SIFT_TEST_PATH}
WORK = WORK_ROOT / "Model3_V8"  # Model 3 keeps its hand-off files and caches in its own folder
WORK.mkdir(parents=True, exist_ok=True)
print(f"DATA: {DATA} | WORK: {WORK}")


# ---------------- Common.py ----------------
# ---------------- metadata ----------------
def LoadMeta():
    """(train, test) DataFrames; missing viewpoint -> 'unknown'."""
    train = pd.read_csv(DATA / "train.csv")
    test = pd.read_csv(DATA / "test.csv")
    for d in (train, test):
        d["viewpoint"] = d["viewpoint"].fillna("unknown")
    return train, test


def IdMaps(train, test):
    """id->species, id->viewpoint, train id->identity"""
    a = pd.concat([train, test], ignore_index=True)
    return dict(zip(a.id, a.species)), dict(zip(a.id, a.viewpoint)), dict(zip(train.id, train.identity))


SIDES = {"left": "L", "frontleft": "L", "backleft": "L",
         "right": "R", "frontright": "R", "backright": "R"}


def Side(vp):
    """L / R flank, O other (front/back/up/down), U unknown"""
    return "U" if vp == "unknown" else SIDES.get(vp, "O")


def ViewRelation(a, b):
    """0 same side, 1 different, 2 opposite flanks (L vs R), 3 unknown"""
    a, b = Side(a), Side(b)
    if "U" in (a, b):
        return 3
    if a == b:
        return 0
    return 2 if {a, b} == {"L", "R"} else 1


# ---------------- features ----------------
def LoadMiewid():
    """(F, Row): F is (9954, 2152) float32, Row[id] -> row index"""
    F = np.load(DATA / "miewid_features.npy").astype(np.float32)
    ids = pd.read_csv(DATA / "miewid_ids.csv")["id"].tolist()
    assert len(ids) == len(F), "miewid ids and features do not line up"
    return F, {i: k for k, i in enumerate(ids)}


EMPTY = (np.zeros((0, 2), np.float32), np.zeros((0, 128), np.uint8))
NPZ = {}


def LoadSift(imgId):
    """(xy (n,2) float32, desc (n,128) uint8); empty arrays if no keypoints.
    Works with an extracted folder of .npy files or the original .npz file.
    Not thread-safe for the .npz case -- load in the main thread."""
    src = SIFT_SRC[imgId[:2]]
    if src.is_dir():
        fx, fd = src / f"{imgId}_xy.npy", src / f"{imgId}_desc.npy"
        if not (fx.exists() and fd.exists()):
            return EMPTY
        xy, d = np.load(fx), np.load(fd)
    else:
        if src not in NPZ:
            NPZ[src] = np.load(src)
        z = NPZ[src]
        if f"{imgId}_xy" not in z.files or f"{imgId}_desc" not in z.files:
            return EMPTY
        xy, d = z[f"{imgId}_xy"], z[f"{imgId}_desc"]
    if len(d) == 0:
        return EMPTY
    return xy.astype(np.float32).reshape(-1, 2), d.astype(np.uint8).reshape(-1, 128)


# ---------------- metric ----------------
def Map5(truth, preds):
    """truth {qid: label}, preds {qid: [up to 5 labels]} -> MAP@5"""
    total = 0.0
    for q, t in truth.items():
        for k, lab in enumerate(preds.get(q, [])[:5]):
            if lab == t:
                total += 1.0 / (k + 1)
                break
    return total / max(len(truth), 1)


# ---------------- validation folds ----------------
def MakeFolds(train, nFolds=4, unkFrac=0.2, qFrac=0.25, cap=10, seed=42, enc=None):
    """Simulate the test set from train. Per fold: hide ~unkFrac of identities entirely
    (their queries' truth = new_individual) and use ~qFrac of the other identities' photos
    as queries. If enc {id: encounter label} is given, whole ENCOUNTERS (bursts of
    near-identical photos) are held out together, so a query never has a near-duplicate
    left in the gallery -- this is what real test photos look like.
    Returns [{fold, queryIds, galleryIds, truth}]"""
    rng = np.random.default_rng(seed)
    lab = dict(zip(train.id, train.identity))
    folds = []
    for f in range(nFolds):
        queries, hidden = [], set()
        for _, d in train.groupby("species"):
            idents = rng.permutation(np.asarray(d["identity"].unique(), dtype=object))
            unk = set(idents[: int(round(unkFrac * len(idents)))])
            for ident, g in d.groupby("identity"):
                ids = g["id"].to_numpy()
                if ident in unk:
                    hidden.update(ids)
                    queries += list(rng.choice(ids, min(cap, len(ids)), replace=False))
                elif enc is None:
                    if len(ids) >= 2:
                        k = min(cap, max(1, int(round(qFrac * len(ids)))), len(ids) - 1)
                        queries += list(rng.choice(ids, k, replace=False))
                else:
                    encs = rng.permutation(np.asarray(sorted({enc[i] for i in ids}), dtype=object))
                    if len(encs) < 2:
                        continue  # a single encounter cannot give a realistic known query
                    k = min(len(encs) - 1, max(1, int(round(qFrac * len(encs)))))
                    qEnc = set(encs[:k])
                    qi = [i for i in ids if enc[i] in qEnc]
                    hidden.update(qi)  # the whole encounter leaves the gallery
                    queries += list(rng.choice(qi, min(cap, len(qi)), replace=False))
        qset = set(queries)
        gallery = [i for i in train.id if i not in qset and i not in hidden]
        galIdents = {lab[i] for i in gallery}
        truth = {q: (lab[q] if lab[q] in galIdents else NEW) for q in sorted(qset)}
        folds.append({"fold": f, "queryIds": sorted(qset), "galleryIds": gallery, "truth": truth})
    return folds


def SavePkl(obj, name):
    with open(WORK / name, "wb") as fh:
        pickle.dump(obj, fh)


def LoadPkl(name, default=None):
    p = WORK / name
    if not p.exists():
        if default is not None:
            return default
        raise FileNotFoundError(f"{p} missing -- has that step been run?")
    with open(p, "rb") as fh:
        return pickle.load(fh)


def LoadFolds():
    return LoadPkl("Folds.pkl")


# ---------------- submission ----------------
def CheckSubmission(df, train, test):
    """AssertionError if Kaggle would reject the file or a label can never score."""
    spIdent = dict(zip(train.identity, train.species))
    spTest = dict(zip(test.id, test.species))
    assert list(df.columns) == ["id", "predictions"], "columns must be id,predictions"
    assert len(df) == len(test) and set(df.id) == set(test.id), "need one row per test id"
    assert not df.id.duplicated().any(), "duplicate ids"
    for i, p in zip(df.id, df.predictions):
        labs = p.split()
        assert 1 <= len(labs) <= 5 and len(set(labs)) == len(labs), f"{i}: bad label list"
        for l in labs:
            assert l == NEW or spIdent.get(l) == spTest[i], f"{i}: bad label {l}"
    return True


def WriteSubmission(preds, test, train, path):
    """preds {test_id: [labels]} -> checked CSV in the project folder"""
    rows = []
    for i in test.id:
        labs = []
        for l in preds.get(i, []):
            if l not in labs:
                labs.append(l)
        rows.append((i, " ".join(labs[:5] or [NEW])))
    df = pd.DataFrame(rows, columns=["id", "predictions"])
    CheckSubmission(df, train, test)
    df.to_csv(OUT_DIR / path, index=False)
    print(f"wrote {path} ({len(df)} rows, format OK)")
    return df